### Authors
Arian Farhanieh & Simon Berg 

### LLM contributions

```
File / notebook           | LLM level | Description
--------------------------|-----------|----------------------------------------------
Project1.ipynb            | 2         | LLM was used to fix some poorly written loops as well as a tool to get ideas and find premade functions such as np.errstate to                        |           |  avoid overflow and invalid warnings. The LLM was also used to help write some docstrings for functions.
```
Additioanlly: OpenAI's ChatGPT was used to verify that some outputs were reasonable, aswell as to get ideas for how to implement some functions. The LLM was also used to help write some docstrings for functions, but many of them are so bad that we wrote most ourselves 

# Project 1 on Machine Learning: regression analysis, resampling methods and gradient descent

**Data Analysis and Machine Learning FYS-STK3155/FYS4155, University of Oslo, fall 2026**

**Deadline: October 5 (midnight), 2026.** Published August 31, 2026.

This notebook is the Jupyter version of the project text (`Project1.pdf`). The
code cells are *starting points*, not solutions: they set up the data and show
the automatic-differentiation check asked for in part e). Everything else is
yours to write.

## Preamble: note on writing reports, using reference material, AI and other tools

We want you to answer the three different projects by handing in reports
written like a standard scientific/technical report. The links at
<https://github.com/EducationalMaterialUiO/MachineLearningUiO/tree/main/doc/Projects>
contain more information. There you can find examples of previous reports, the
projects themselves, how we grade reports etc. How to write reports will also be
discussed during the various lab sessions. Please do ask us if you are in doubt.
In that folder you will find how we evaluate the projects. You will receive a
feedback with a final score for each project.

When using codes and material from other sources, you should refer to these in
the bibliography of your report, indicating wherefrom you for example got the
code, whether this is from the lecture notes, software like **Scikit-Learn**,
**JAX**, **TensorFlow**, **PyTorch** or other sources. These sources should
always be cited correctly. How to cite some of the libraries is often indicated
from their corresponding GitHub sites or websites; see for example how to cite
Scikit-Learn at <https://scikit-learn.org/dev/about.html>.

We encourage you to use Large Language models (LLMs) like
[ChatGPT](https://openai.com/chatgpt/), [Claude](https://claude.ai) or similar
in writing the report and in developing your code. In the report, you should
after the conclusions, write a declaration on how you have used LLMs following
the instructions at
<https://github.com/EducationalMaterialUiO/MachineLearningUiO/blob/main/LLM_Usage_Declaration_Guidelines.md>.
Failing to do so, <u>leads to an automatic deduction of five (5) points out of
the final score of 100 for the project</u>. Remember that *you* are responsible
for every line of code and every claim in the report: be prepared to explain and
defend all of it. If you use LLMs, please take a look at the guidelines at
<https://github.com/EducationalMaterialUiO/MachineLearningUiO/tree/main/doc/UsingLLMs>
on how to use these tools efficiently.

If you would like to study other data sets, feel free to propose other sets.
What we have proposed here are mere suggestions from our side. If you opt for
another data set, consider using a set which has been studied in the scientific
literature. This makes it easier for you to compare and analyze your results.
Comparing with existing results from the scientific literature is also an
essential element of the scientific discussion. The University of California at
Irvine with its Machine Learning repository at
<https://archive.ics.uci.edu/ml/index.php> is an excellent site to look up for
examples and inspiration. [Kaggle.com](https://www.kaggle.com/) is an equally
interesting site. Feel free to explore these sites and/or ask us for inputs
relevant to your discipline. When selecting other data sets, make sure these
are sets used for regression problems (not classification).

## Regression analysis, resampling methods and gradient descent

The main aim of this project is to study in more detail various regression
methods, including Ordinary Least Squares (OLS) regression, Ridge regression and
Lasso regression, together with the two resampling techniques (the bootstrap and
cross-validation) that let us assess and select models, and the gradient descent
methods that will be our optimizers for the rest of the course. In addition to
the scientific part, in this course we want also to give you an experience in
writing scientific reports.

We will study how to fit polynomials to a specific one-dimensional function
(feel free to replace the suggested function with more complicated ones). We
will use Runge's function (see
<https://en.wikipedia.org/wiki/Runge%27s_phenomenon> for a discussion),

$$
f(x) = \frac{1}{1+25x^2}, \qquad x \in [-1, 1].
$$

The project is organized so that it follows the weekly exercises and lectures of
the first weeks of the course:

- **Parts a) and b):** OLS and Ridge regression with closed-form solutions
  (weeks 34–36; the exercise sets of weeks 35 and 36 can be reused directly).
- **Parts c) and d):** the bias-variance tradeoff studied with the bootstrap,
  and cross-validation (week 36; Chapter 2, Sections 2.9–2.12 of the lecture
  notes).
- **Parts e) and f):** replacing the closed-form solutions by gradient descent,
  with gradients computed both analytically and by automatic differentiation,
  and with momentum and adaptive learning rates (weeks 37–38; Chapter 4).
- **Parts g) and h):** Lasso regression, our first method without a closed-form
  solution, and stochastic gradient descent (Chapter 4 of the lecture notes).
- **Part i):** the final comparison of OLS, Ridge and Lasso using
  cross-validation.

The analytical parts (in particular the derivation of the bias-variance
decomposition in part c) belong in the theory section of your report.
The material for solving the various exercises is covered by Chapters 2–4 of
the lecture notes and the lecture material for weeks 34–38.

### Setting up the data

A starting point: Runge's function on $[-1,1]$ with additive Gaussian noise,
and a polynomial design matrix. Feel free to change everything.

In [ ]:
# parts a)-d): FYSSTK4155_Project1_Part_abcd.py; plot_style from the project starter notebook, cell 5
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.model_selection import train_test_split, KFold

# Tool: OpenAI's ChatGPT
# Role: used to find the names of the callables styles
def plot_style():
    plt.style.use("seaborn-v0_8-darkgrid")
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 14,
        "axes.labelsize": 16,
        "axes.titlesize": 18,
        "legend.fontsize": 12,
        "figure.figsize": (8,6),
        "axes.grid": True,
        "grid.alpha": 0.9,
        "grid.linestyle": "solid",
        "text.usetex": True,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "xtick.major.size": 6,
        "xtick.minor.size": 3,
        "ytick.major.size": 6,
        "ytick.minor.size": 3,
        "xtick.major.width": 1.0,
        "xtick.minor.width": 0.75,
        "ytick.major.width": 1.0,
        "ytick.minor.width": 0.75,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.color": "black",
        "ytick.color": "black",
    })


plot_style()
legend_style = dict(frameon=True, facecolor="lightgray", edgecolor="gray", framealpha=0.7)
markers      = ("o", "x", "^", "v", "d", "s")
os.makedirs("proj1_figures", exist_ok=True)


"""
Functions shared by all parts
"""

def runge_function(x):
    return 1 / (1 + 25 * x**2)


def sci_notation(x):
    """Scientific notation for plot labels"""
    const, exp = f"{x:.1e}".split("e")
    return rf"{const} \times10^{{{int(exp)}}}"


def design_matrix(x, p):
    """Columns x, x^2, ..., x^p, no intercept column"""
    X = np.zeros((len(x), p))
    for i in range(1, p + 1):
        X[:, i - 1] = x**i
    return X


def MSE(y, y_prediction):
    return np.mean((y - y_prediction)**2)


def R_squared(y, y_prediction):
    return 1 - np.sum((y - y_prediction)**2) / np.sum((y - np.mean(y))**2)

In [ ]:
# data used in parts e)-i)
rng   = np.random.default_rng(2026)
n     = 100
sigma = 0.1
x     = np.sort(rng.uniform(-1, 1, n))
y     = runge_function(x) + rng.normal(0, sigma, n)

xx = np.linspace(-1, 1, 400)
# Role: used again to find these cool colors for the plot
plt.plot(xx, runge_function(xx), color="#004488", label="Runge's function")
plt.scatter(x, y, s=12, color="#BB5566", label=rf"data, $\sigma={sigma}$")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(**legend_style)
plt.show()

### Part a): Ordinary Least Squares (OLS) for the Runge function

We will generate our own data set for the above function $f(x)$ with
$x\in[-1,1]$, using either a uniform distribution or a fixed step size to set up
the array of $x$ values. You should also explore the addition of normally
distributed noise $\varepsilon\sim\mathcal{N}(0,\sigma^2)$ to the function; a
noise level of $\sigma\approx0.1$ is a good starting point, but study how your
conclusions depend on it.

**Write your own code** (using for example the pseudoinverse function `pinv`
from `numpy`, or the SVD directly) and perform a standard **ordinary least
squares regression** analysis using polynomials in $x$ up to order 15 or higher.
Explore the dependence on the number of data points and the polynomial degree.

Evaluate the mean squared error (MSE)

$$
\mathrm{MSE}(\boldsymbol{y},\tilde{\boldsymbol{y}}) = \frac{1}{n}\sum_{i=0}^{n-1}(y_i-\tilde{y}_i)^2,
$$

and the $R^2$ score function. If $\tilde{y}_i$ is the predicted value of the
$i$-th sample and $y_i$ is the corresponding true value, then the score $R^2$ is
defined as

$$
R^2(\boldsymbol{y},\tilde{\boldsymbol{y}}) = 1 - \frac{\sum_{i=0}^{n-1}(y_i-\tilde{y}_i)^2}{\sum_{i=0}^{n-1}(y_i-\bar{y})^2},
\qquad
\bar{y} = \frac{1}{n}\sum_{i=0}^{n-1}y_i .
$$

Plot the resulting scores (MSE and $R^2$) as functions of the polynomial degree
(here up to polynomial degree 15). Plot also the parameters $\boldsymbol{\theta}$
as you increase the order of the polynomial. Comment your results.

Your code has to include a scaling/centering of the data (for example by
subtracting the mean value), and a split of the data in training and test data.
For the splitting you can either write your own code or use for example the
function `train_test_split` provided by **Scikit-Learn** (make sure you have
installed it). **You should present a critical discussion of why and how you
have scaled/ or not scaled the data** (see Section 3.13 of the lecture notes,
the Tuesday session of week 35 and the pertinent notebook for week 35).

It is normal in essentially all Machine Learning studies to split the data in a
training set and a test set (eventually also an additional validation set).
There is no explicit recipe for how much data should be included as training
data and say test data. An accepted rule of thumb is to use approximately $2/3$
to $4/5$ of the data as training data.

You can easily reuse the solutions to your exercises from week 35 (the
analytical derivatives of the cost function, your own OLS code, and the
train/test analysis). See also the lecture slides from weeks 34 and 35 and
Chapter 3 of the lecture notes. On scaling, we recommend reading the following
section from the Scikit-Learn documentation:
<https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html>.
See also Section 3.13 of the lecture notes.

In [ ]:
def generate_data(n, sigma, seed=676767):
    """Noisy Runge data on an even grid in [-1, 1], split 75/25 into training and test data"""
    rng_data = np.random.default_rng(seed)

    x_data = np.linspace(-1, 1, n)
    y_data = runge_function(x_data) + rng_data.normal(0, sigma, n)

    return train_test_split(x_data, y_data, test_size=0.25, random_state=seed)


def OLS(n, sigma, highest_degree, seed):

    x_train, x_test, y_train, y_test = generate_data(n, sigma, seed)

    """
    Arrays for storing results
    """
    polynomial_degrees = np.arange(1, highest_degree + 1)

    mse_train    = np.zeros(highest_degree)
    mse_test     = np.zeros(highest_degree)
    r2_train     = np.zeros(highest_degree)
    r2_test      = np.zeros(highest_degree)
    theta_matrix = np.zeros((highest_degree, highest_degree + 1))   # row i: theta_0, ..., theta_p for p = i + 1

    """
    OLS for every polynomial degree
    """
    for i, degree in enumerate(polynomial_degrees):

        X_train = design_matrix(x_train, degree)
        X_test  = design_matrix(x_test, degree)

        # centring with the training mean
        X_mean = np.mean(X_train, axis=0)
        y_mean = np.mean(y_train)

        X_train_centered = X_train - X_mean
        X_test_centered  = X_test - X_mean
        y_train_centered = y_train - y_mean

        theta_OLS   = np.linalg.pinv(X_train_centered) @ y_train_centered
        theta_0_OLS = y_mean - X_mean @ theta_OLS

        theta_matrix[i, 0]            = theta_0_OLS
        theta_matrix[i, 1:degree + 1] = theta_OLS

        y_train_pred = X_train_centered @ theta_OLS + y_mean
        y_test_pred  = X_test_centered @ theta_OLS + y_mean

        mse_train[i] = MSE(y_train, y_train_pred)
        mse_test[i]  = MSE(y_test, y_test_pred)
        r2_train[i]  = R_squared(y_train, y_train_pred)
        r2_test[i]   = R_squared(y_test, y_test_pred)

    return polynomial_degrees, mse_train, mse_test, r2_train, r2_test, theta_matrix


OLS_result = OLS(n=100, sigma=0.1, highest_degree=15, seed=676767)

In [ ]:
def plot_MSE_R2_OLS_with_thetaj(OLS_result):
    polynomial_degrees, mse_train, mse_test, r2_train, r2_test, theta_matrix = OLS_result

    print(f"OLS: min test MSE {np.min(mse_test)} (train {np.min(mse_train)}) at degree "
          f"{polynomial_degrees[np.argmin(mse_test)]}, max test R2 {np.max(r2_test)}")

    fig      = plt.figure(figsize=(10, 8))
    gridspec = fig.add_gridspec(2, 2, height_ratios=[1, 1])

    ax1 = plt.subplot(gridspec[0, 0])
    ax1.plot(polynomial_degrees, mse_train, "o-", label="MSE train", color="lightblue")
    ax1.plot(polynomial_degrees, mse_test, "x-", label="MSE test", color="blue")
    ax1.plot(polynomial_degrees, np.zeros_like(polynomial_degrees), "--", label="MSE = 0", color="gray", alpha=0.5)
    ax1.plot(polynomial_degrees, 0.01 * np.ones_like(polynomial_degrees), "--", label="MSE = 0.01, irreducible noise", color="black", alpha=0.5)
    ax1.set_xlabel("Polynomial degree")
    ax1.set_ylabel("MSE")
    ax1.legend(**legend_style)

    ax2 = plt.subplot(gridspec[0, 1])
    ax2.plot(polynomial_degrees, r2_train, "o-", label=r"$R^2$ train", color="red")
    ax2.plot(polynomial_degrees, r2_test, "x-", label=r"$R^2$ test", color="crimson")
    ax2.plot(polynomial_degrees, np.ones_like(polynomial_degrees), "--", label=r"$R^2 = 1$", color="black", alpha=0.5)
    ax2.set_xlabel("Polynomial degree")
    ax2.set_ylabel("$R^2$")
    ax2.legend(**legend_style)

    ax3 = plt.subplot(gridspec[1, :])
    for j in range(theta_matrix.shape[1]):
        ax3.plot(polynomial_degrees, theta_matrix[:, j], "o-", label=f"$\\theta_{{{j}}}$")
    ax3.set_xlabel("Polynomial degree")
    ax3.set_ylabel(r"$\theta_j$")
    ax3.legend(loc="upper left", bbox_to_anchor=(1, 1), **legend_style)

    plt.tight_layout()
    plt.savefig("proj1_figures/MSE_R2_thetaj_OLS_a).pdf", format="pdf")
    plt.show()


plot_MSE_R2_OLS_with_thetaj(OLS_result)

### Part b): Adding Ridge regression for the Runge function

Write your own code for the Ridge method, as done in the previous part. The
lecture notes from weeks 35 and 36 contain more information, as well as your
exercises from weeks 35 and 36. Feel free to reuse this material.

Perform the same analysis as you did in the previous part but now for different
values of the penalty parameter $\lambda$. Compare and analyze your results with
those obtained in part a) with the OLS method. Study the dependence on
$\lambda$, and connect what you see to the shrinkage of the singular-value modes
discussed in the lectures (Section 3.8 of the lecture notes).

In [ ]:
def Ridge(n, sigma, highest_degree, seed, n_lambdas=80):

    x_train, x_test, y_train, y_test = generate_data(n, sigma, seed)

    """
    Arrays for storing results
    """
    polynomial_degrees = np.arange(1, highest_degree + 1)
    lambda_parameters  = np.logspace(-9, 9, n_lambdas)

    MSE_train_ridge = np.zeros((highest_degree, n_lambdas))
    MSE_test_ridge  = np.zeros((highest_degree, n_lambdas))
    R2_train_ridge  = np.zeros((highest_degree, n_lambdas))
    R2_test_ridge   = np.zeros((highest_degree, n_lambdas))

    # [i, j, k]: degree i + 1, lambda j, theta_k
    theta_ridge_history = np.zeros((highest_degree, n_lambdas, highest_degree + 1))

    for i, degree in enumerate(polynomial_degrees):

        X_train = design_matrix(x_train, degree)
        X_test  = design_matrix(x_test, degree)

        X_mean = np.mean(X_train, axis=0)
        y_mean = np.mean(y_train)

        X_train_centered = X_train - X_mean
        X_test_centered  = X_test - X_mean
        y_train_centered = y_train - y_mean

        I = np.eye(degree)

        for j, lambda_val in enumerate(lambda_parameters):

            theta_Ridge   = np.linalg.pinv(X_train_centered.T @ X_train_centered + lambda_val * I) @ X_train_centered.T @ y_train_centered
            theta_0_Ridge = y_mean - X_mean @ theta_Ridge

            theta_ridge_history[i, j, 0]            = theta_0_Ridge
            theta_ridge_history[i, j, 1:degree + 1] = theta_Ridge

            y_train_pred_ridge = X_train_centered @ theta_Ridge + y_mean
            y_test_pred_ridge  = X_test_centered @ theta_Ridge + y_mean

            MSE_train_ridge[i, j] = MSE(y_train, y_train_pred_ridge)
            MSE_test_ridge[i, j]  = MSE(y_test, y_test_pred_ridge)
            R2_train_ridge[i, j]  = R_squared(y_train, y_train_pred_ridge)
            R2_test_ridge[i, j]   = R_squared(y_test, y_test_pred_ridge)

    return polynomial_degrees, MSE_train_ridge, MSE_test_ridge, R2_train_ridge, R2_test_ridge, theta_ridge_history, lambda_parameters


ridge_result = Ridge(n=100, sigma=0.1, highest_degree=15, seed=676767)

In [ ]:
def plot_MSE_OLS_thetaj_Ridge_selected_lambdas(ridge_result, OLS_result, degree, selected_lambdas=(1e-8, 1e-4, 1, 1e4, 1e8)):

    polynomial_degrees, mse_train, mse_test, r2_train, r2_test, theta_matrix = OLS_result
    polynomial_degrees, MSE_train_ridge, MSE_test_ridge, R2_train_ridge, R2_test_ridge, theta_ridge_history, lambda_parameters = ridge_result

    degree_idx = np.where(polynomial_degrees == degree)[0][0]

    fig      = plt.figure(figsize=(10, 8))
    gridspec = fig.add_gridspec(2, 2, height_ratios=[1, 1])

    ax1 = plt.subplot(gridspec[0, 0])
    ax1.plot(polynomial_degrees, mse_test, "x--", label="MSE test OLS", color="blue")
    for lambdas in selected_lambdas:
        j = np.argmin(np.abs(lambda_parameters - lambdas))
        ax1.plot(polynomial_degrees, MSE_test_ridge[:, j], "o--", label=fr"Ridge $\lambda={sci_notation(lambda_parameters[j])}$")
    ax1.set_xlabel("Polynomial degree")
    ax1.set_ylabel("MSE test")
    ax1.legend(loc="upper right", **legend_style)

    ax2 = plt.subplot(gridspec[0, 1])
    ax2.semilogx(lambda_parameters, MSE_train_ridge[degree_idx, :], "o-", label="MSE train", color="lightgreen")
    ax2.semilogx(lambda_parameters, MSE_test_ridge[degree_idx, :], "x-", label="MSE test", color="green")
    ax2.plot(lambda_parameters, np.zeros_like(lambda_parameters), "--", label="MSE = 0", color="black", alpha=0.5)
    ax2.set_xlabel(r"$\lambda$")
    ax2.set_ylabel(fr"MSE Ridge, p={degree}")
    ax2.legend(**legend_style)

    ax3 = plt.subplot(gridspec[1, 0])
    ax3.semilogx(lambda_parameters, R2_train_ridge[degree_idx, :], "o-", label=r"$R^2$ train", color="orange")
    ax3.semilogx(lambda_parameters, R2_test_ridge[degree_idx, :], "x-", label=r"$R^2$ test", color="brown")
    ax3.plot(lambda_parameters, np.ones_like(lambda_parameters), "--", label=r"$R^2 = 1$", color="black", alpha=0.5)
    ax3.set_xlabel(r"$\lambda$")
    ax3.set_ylabel(fr"$R^2$ Ridge, p={degree}")
    ax3.legend(**legend_style)

    ax4 = plt.subplot(gridspec[1, 1])
    for j in range(degree + 1):
        ax4.semilogx(lambda_parameters, theta_ridge_history[degree_idx, :, j], label=fr"$\theta_{{{j}}}$")
    ax4.set_xlabel(r"$\lambda$")
    ax4.set_ylabel(fr"$\theta_j, p={degree}$")
    ax4.legend(loc="upper right", ncol=2, bbox_to_anchor=(1, 1), **legend_style)

    plt.tight_layout()
    plt.savefig(f"proj1_figures/MSE_R2_thetaj_OLS_Ridge_selected_lambdas_p={degree}_b).pdf", format="pdf")
    plt.show()


def plot_thetaj_lambda(ridge_result, degrees_to_plot=(5, 10, 15)):

    polynomial_degrees, MSE_train_ridge, MSE_test_ridge, R2_train_ridge, R2_test_ridge, theta_ridge_history, lambda_parameters = ridge_result

    for degree in degrees_to_plot:
        degree_idx = np.where(polynomial_degrees == degree)[0][0]

        plt.figure()
        for j in range(degree + 1):
            plt.semilogx(lambda_parameters, theta_ridge_history[degree_idx, :, j], label=fr"$\theta_{{{j}}}$")

        plt.xlabel(r"$\lambda$")
        plt.ylabel(fr"$\theta_j, p={degree}$")
        plt.xlim(1e-9, 1e9)
        plt.legend(loc="upper right", bbox_to_anchor=(1, 1), **legend_style)
        plt.tight_layout()
        plt.savefig(f"proj1_figures/thetaj_lambda_Ridge_p={degree}_b).pdf", format="pdf")
        plt.show()


plot_MSE_OLS_thetaj_Ridge_selected_lambdas(ridge_result, OLS_result, degree=15)
plot_thetaj_lambda(ridge_result)

### Part c): The bias-variance tradeoff and the bootstrap

Our aim here is to study the bias-variance tradeoff by implementing the
**bootstrap** resampling technique. **We will only use the simpler ordinary
least squares here.**

Before you perform an analysis of the bias-variance tradeoff on your test data,
make first a figure similar to Fig. 2.11 of Hastie, Tibshirani and Friedman.
Figure 2.11 of this reference displays only the test and training MSEs. The
test MSE can be used to indicate possible regions of low/high bias and variance.
You will most likely not get an equally smooth curve! You will also need to
increase the polynomial order and play around with the number of data points as
well (see the exercise sets from week 35 and week 36).

With this result we move on to the bias-variance tradeoff analysis. Consider a
data set $\mathcal{L}$ consisting of the data
$\boldsymbol{X}_\mathcal{L}=\{(y_j,\boldsymbol{x}_j),\,j=0,\dots,n-1\}$, and
assume that the true data are generated from a noisy model

$$
\boldsymbol{y} = f(\boldsymbol{x}) + \boldsymbol{\varepsilon},
$$

where $\boldsymbol{\varepsilon}$ is normally distributed with mean zero and
variance $\sigma^2$. In our derivation of the ordinary least squares method we
defined an approximation to the function $f$ in terms of the parameters
$\boldsymbol{\theta}$ and the design matrix $\boldsymbol{X}$ which embody our
model, that is $\tilde{\boldsymbol{y}}=\boldsymbol{X}\boldsymbol{\theta}$. The
parameters $\boldsymbol{\theta}$ are in turn found by optimizing the mean squared
error via the cost function

$$
C(\boldsymbol{X},\boldsymbol{\theta}) = \frac{1}{n}\sum_{i=0}^{n-1}(y_i-\tilde{y}_i)^2
 = \mathbb{E}\left[(\boldsymbol{y}-\tilde{\boldsymbol{y}})^2\right].
$$

Here the expected value $\mathbb{E}$ is the sample value.

Show that you can rewrite this in terms of a term which contains the variance of
the model itself (the so-called variance term), a term which measures the
deviation from the true data and the mean value of the model (the bias term) and
finally the variance of the noise. That is, show that

$$
\mathbb{E}\left[(\boldsymbol{y}-\tilde{\boldsymbol{y}})^2\right]
 = \mathrm{Bias}^2[\tilde{y}] + \mathrm{var}[\tilde{y}] + \sigma^2,
$$

with

$$
\mathrm{Bias}^2[\tilde{y}] = \mathbb{E}\left[\left(\boldsymbol{f}-\mathbb{E}[\tilde{\boldsymbol{y}}]\right)^2\right],
\qquad
\mathrm{var}[\tilde{y}] = \mathbb{E}\left[\left(\tilde{\boldsymbol{y}}-\mathbb{E}[\tilde{\boldsymbol{y}}]\right)^2\right]
 = \frac{1}{n}\sum_i\left(\tilde{y}_i-\mathbb{E}[\tilde{\boldsymbol{y}}]\right)^2 .
$$

**Important note:** since the function $f(x)$ is unknown, in order to be able to
evaluate the bias we replace $f(\boldsymbol{x})$ in the expression for the bias
with $\boldsymbol{y}$; discuss what this does to the measured bias term (it
absorbs the noise variance $\sigma^2$). Explain what the three terms mean and
discuss their interpretations, and state clearly over which randomness the
expectation values are taken.

The answer to this exercise should be included in the theory part of the
report. This derivation is also part of the weekly exercises of week 36, and it
is Eq. (2.52) of Chapter 2 of the lecture notes.

Perform then a bias-variance analysis of the Runge function by studying the MSE
value as function of the complexity of your model, using the **bootstrap**
resampling method to estimate the expectation values over training sets.
Discuss the bias and variance tradeoff as function of your model complexity (the
degree of the polynomial) and the number of data points. You can follow the
code examples in Sections 2.10 and 2.11 of the lecture notes and the Tuesday
notebook of week 36 (`week36tuesday.ipynb`).

In [ ]:
def bootstrap_bias_variance(n, sigma, highest_degree, n_bootstraps, seed):

    x_train, x_test, y_train, y_test = generate_data(n, sigma, seed)

    rng_bootstrap      = np.random.default_rng(seed)
    polynomial_degrees = np.arange(1, highest_degree + 1)

    error                 = np.zeros(highest_degree)
    bias_squared          = np.zeros(highest_degree)
    bias_squared_measured = np.zeros(highest_degree)
    variance              = np.zeros(highest_degree)

    fx_test = runge_function(x_test)

    for i, degree in enumerate(polynomial_degrees):

        y_pred_bootstrap = np.zeros((len(x_test), n_bootstraps))

        for j in range(n_bootstraps):

            # resample the training data with replacement
            idx               = rng_bootstrap.choice(len(x_train), len(x_train), replace=True)
            x_train_bootstrap = x_train[idx]
            y_train_bootstrap = y_train[idx]

            X_train_bootstrap = design_matrix(x_train_bootstrap, degree)
            X_test_bootstrap  = design_matrix(x_test, degree)

            X_mean = np.mean(X_train_bootstrap, axis=0)
            y_mean = np.mean(y_train_bootstrap)

            X_train_bootstrap_centered = X_train_bootstrap - X_mean
            X_test_bootstrap_centered  = X_test_bootstrap - X_mean
            y_train_bootstrap_centered = y_train_bootstrap - y_mean

            theta_bootstrap        = np.linalg.pinv(X_train_bootstrap_centered) @ y_train_bootstrap_centered
            y_pred_bootstrap[:, j] = X_test_bootstrap_centered @ theta_bootstrap + y_mean

        total_mean = np.mean(y_pred_bootstrap, axis=1)

        bias_squared[i]          = np.mean((fx_test - total_mean)**2)
        bias_squared_measured[i] = np.mean((y_test - total_mean)**2)
        variance[i]              = np.mean(np.var(y_pred_bootstrap, axis=1))
        error[i]                 = np.mean((y_test[:, np.newaxis] - y_pred_bootstrap)**2)   # every bootstrap prediction against the same y_i

    print(f"n = {n}: max |error - (bias^2_measured + variance)| = {np.max(np.abs(error - bias_squared_measured - variance))}")

    return polynomial_degrees, error, bias_squared, bias_squared_measured, variance, n


OLS_result_p30        = OLS(n=100, sigma=0.1, highest_degree=30, seed=676767)
bootstrap_result_n50  = bootstrap_bias_variance(n=50, sigma=0.1, highest_degree=15, n_bootstraps=500, seed=67676)
bootstrap_result_n100 = bootstrap_bias_variance(n=100, sigma=0.1, highest_degree=15, n_bootstraps=500, seed=67676)
bootstrap_result_n300 = bootstrap_bias_variance(n=300, sigma=0.1, highest_degree=15, n_bootstraps=500, seed=67676)

In [ ]:
def plot_traintest_MSE_reproduce_fig2_11(OLS_result):

    polynomial_degrees, mse_train, mse_test, r2_train, r2_test, theta_matrix = OLS_result

    plt.figure()
    plt.semilogy(polynomial_degrees, mse_test, "-", label="MSE test OLS", color="red")
    plt.semilogy(polynomial_degrees, mse_train, "-", label="MSE train OLS", color="cyan")

    plt.legend(loc="upper right", ncol=2, bbox_to_anchor=(1, 1), **legend_style)
    plt.xlabel("Polynomial degree")
    plt.ylabel("MSE")
    plt.tight_layout()
    plt.savefig(f"proj1_figures/traintest_MSE_reproduce_fig2_11_c)_p={polynomial_degrees[-1]}.pdf", format="pdf")
    plt.show()


def plot_comparison_bias_variance(bootstrap_result):

    polynomial_degrees, error, bias_squared, bias_squared_measured, variance, n = bootstrap_result

    plt.figure()
    plt.semilogy(polynomial_degrees, error, "o-", label="Error/MSE", color="blue")
    plt.semilogy(polynomial_degrees, bias_squared, "x-", label=r"Bias$^2$", color="red")
    plt.semilogy(polynomial_degrees, bias_squared_measured, "^-", label=r"Bias$^2_{\mathrm{measured}}$", color="orange")
    plt.semilogy(polynomial_degrees, variance, "s-", label="Variance", color="green")

    plt.xlabel("Polynomial degree")
    plt.ylabel(fr"Error, bias$^2$, variance, n={n}")
    plt.legend(**legend_style)
    plt.tight_layout()
    plt.savefig(f"proj1_figures/bias_variance_comparisons_n={n}_c).pdf", format="pdf")
    plt.show()


def plot_comparison_different_ns(bootstrap_results):
    """bootstrap_results: results for n = 50, 100 and 300"""

    colors = {"error": ("cyan", "blue", "navy"), "variance": ("lightgreen", "green", "darkgreen"), "bias": ("pink", "orange", "red")}

    fig      = plt.figure(figsize=(10, 8))
    gridspec = fig.add_gridspec(2, 2, height_ratios=[1, 1])
    ax1      = plt.subplot(gridspec[0, 0])
    ax2      = plt.subplot(gridspec[0, 1])
    ax3      = plt.subplot(gridspec[1, :])

    for k, (polynomial_degrees, error, bias_squared, bias_squared_measured, variance, n) in enumerate(bootstrap_results):
        marker = ("o", "x", "^")[k]
        ax1.semilogy(polynomial_degrees, error, marker + "-", label=f"n={n}", color=colors["error"][k])
        ax2.semilogy(polynomial_degrees, variance, marker + "-", label=f"Variance n={n}", color=colors["variance"][k])
        ax3.semilogy(polynomial_degrees, bias_squared, marker + "-", label=fr"Bias$^2$ n={n}", color=colors["bias"][k])
        ax3.semilogy(polynomial_degrees, bias_squared_measured, marker + "--", label=fr"Bias$^2_{{\mathrm{{measured}}}}$ n={n}", color=colors["bias"][k])

    ax1.set_xlabel("Polynomial degree")
    ax1.set_ylabel("MSE")
    ax1.legend(loc="upper left", **legend_style)

    ax2.set_xlabel("Polynomial degree")
    ax2.set_ylabel("Variance")
    ax2.legend(loc="upper left", **legend_style)

    ax3.set_xlabel("Polynomial degree")
    ax3.set_ylabel(r"Bias$^2$")
    ax3.legend(loc="upper center", ncol=3, **legend_style)

    plt.tight_layout()
    plt.savefig("proj1_figures/bias_variance_comparisons_different_ns_c).pdf", format="pdf")
    plt.show()


plot_traintest_MSE_reproduce_fig2_11(OLS_result_p30)
plot_comparison_bias_variance(bootstrap_result_n100)
plot_comparison_different_ns((bootstrap_result_n50, bootstrap_result_n100, bootstrap_result_n300))

### Part d): Cross-validation as a resampling technique

The aim here is to use another widely popular resampling technique, the
so-called cross-validation method (Section 2.12 of the lecture notes and the
Tuesday session of week 36). For Ridge and Lasso, see also Section 3.14 of the
lecture notes.

Use the $k$-fold cross-validation functionality of **Scikit-Learn** (`KFold`
together with `cross_val_score`, or `cross_validate`) and evaluate again the
MSE function resulting from the test folds, for the OLS analysis of parts a)
and c) as a function of the polynomial degree, and for Ridge regression as a
function of both the polynomial degree and $\lambda$. Try $k=5$ and $k=10$.
Optionally, you can also write your own $k$-fold cross-validation code and
check that it reproduces the Scikit-Learn results when the two use the same
fold assignment (the Tuesday session of week 36 shows how).

Compare the MSE you get from cross-validation with the one you got
from your **bootstrap** code in part c). Do the two methods point to the same
model complexity? Comment and interpret your results. Make sure that any scaling
or other preprocessing that learns from the data is performed *inside* the
cross-validation loop (fitted on the training folds only), and explain why this
matters.

In [ ]:
def OLS_cross_validation(n, sigma, highest_degree, k, seed):

    # the full data set, split into folds below
    x_train, x_test, y_train, y_test = generate_data(n, sigma, seed)
    x_full = np.concatenate((x_train, x_test))
    y_full = np.concatenate((y_train, y_test))

    polynomial_degrees = np.arange(1, highest_degree + 1)
    MSE_CrossVal       = np.zeros(highest_degree)
    MSE_CrossVal_std   = np.zeros(highest_degree)

    kfold = KFold(n_splits=k, shuffle=True, random_state=seed)

    for i, degree in enumerate(polynomial_degrees):

        X         = design_matrix(x_full, degree)
        MSE_folds = []

        for train_index, val_index in kfold.split(X):

            X_train_fold, X_test_fold = X[train_index], X[val_index]
            y_train_fold, y_test_fold = y_full[train_index], y_full[val_index]

            # centring learnt from the training fold only
            X_mean = np.mean(X_train_fold, axis=0)
            y_mean = np.mean(y_train_fold)

            X_train_fold_centered = X_train_fold - X_mean
            X_test_fold_centered  = X_test_fold - X_mean
            y_train_fold_centered = y_train_fold - y_mean

            theta_OLS_fold   = np.linalg.pinv(X_train_fold_centered) @ y_train_fold_centered
            y_test_pred_fold = X_test_fold_centered @ theta_OLS_fold + y_mean

            MSE_folds.append(MSE(y_test_fold, y_test_pred_fold))

        MSE_CrossVal[i]     = np.mean(MSE_folds)
        MSE_CrossVal_std[i] = np.std(MSE_folds)

    return polynomial_degrees, MSE_CrossVal, MSE_CrossVal_std


def Ridge_cross_validation(n, sigma, highest_degree, k, n_lambdas, seed):

    x_train, x_test, y_train, y_test = generate_data(n, sigma, seed)
    x_full = np.concatenate((x_train, x_test))
    y_full = np.concatenate((y_train, y_test))

    polynomial_degrees = np.arange(1, highest_degree + 1)
    lambda_parameters  = np.logspace(-9, 9, n_lambdas)

    MSE_CrossVal_Ridge     = np.zeros((highest_degree, n_lambdas))
    MSE_CrossVal_Ridge_std = np.zeros((highest_degree, n_lambdas))

    kfold = KFold(n_splits=k, shuffle=True, random_state=seed)

    for i, degree in enumerate(polynomial_degrees):

        X = design_matrix(x_full, degree)
        I = np.eye(degree)
        fold_errors = np.zeros((k, n_lambdas))

        for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X)):

            X_train_fold, X_test_fold = X[train_idx], X[test_idx]
            y_train_fold, y_test_fold = y_full[train_idx], y_full[test_idx]

            X_mean = np.mean(X_train_fold, axis=0)
            y_mean = np.mean(y_train_fold)

            X_train_fold_centered = X_train_fold - X_mean
            X_test_fold_centered  = X_test_fold - X_mean
            y_train_fold_centered = y_train_fold - y_mean

            # the same for every lambda
            XTX = X_train_fold_centered.T @ X_train_fold_centered
            XTy = X_train_fold_centered.T @ y_train_fold_centered

            for j, lambda_val in enumerate(lambda_parameters):
                theta_Ridge_fold         = np.linalg.pinv(XTX + lambda_val * I) @ XTy
                y_test_pred_fold         = X_test_fold_centered @ theta_Ridge_fold + y_mean
                fold_errors[fold_idx, j] = MSE(y_test_fold, y_test_pred_fold)

        MSE_CrossVal_Ridge[i, :]     = np.mean(fold_errors, axis=0)
        MSE_CrossVal_Ridge_std[i, :] = np.std(fold_errors, axis=0)

    return polynomial_degrees, lambda_parameters, MSE_CrossVal_Ridge, MSE_CrossVal_Ridge_std


OLS_CrossValK5_result    = OLS_cross_validation(n=100, sigma=0.1, highest_degree=15, k=5, seed=676767)
OLS_CrossValK10_result   = OLS_cross_validation(n=100, sigma=0.1, highest_degree=15, k=10, seed=676767)
Ridge_CrossValK5_result  = Ridge_cross_validation(n=100, sigma=0.1, highest_degree=15, k=5, n_lambdas=80, seed=67676)
Ridge_CrossValK10_result = Ridge_cross_validation(n=100, sigma=0.1, highest_degree=15, k=10, n_lambdas=80, seed=67676)

In [ ]:
def plot_OLS_CrossVal(CrossValK5, CrossValK10, bootstrap_result):

    degrees5, MSE_CrossVal_K5, MSE_CrossVal_std_K5    = CrossValK5
    degrees10, MSE_CrossVal_K10, MSE_CrossVal_std_K10 = CrossValK10
    bootstrap_degrees, bootstrap_error                = bootstrap_result[0], bootstrap_result[1]

    print(f"best degree: K=5 {degrees5[np.argmin(MSE_CrossVal_K5)]} ({np.min(MSE_CrossVal_K5)}), "
          f"K=10 {degrees10[np.argmin(MSE_CrossVal_K10)]} ({np.min(MSE_CrossVal_K10)}), "
          f"bootstrap {bootstrap_degrees[np.argmin(bootstrap_error)]} ({np.min(bootstrap_error)})")

    plt.figure()
    plt.errorbar(degrees5, MSE_CrossVal_K5, yerr=MSE_CrossVal_std_K5, fmt="o-", label="MSE Cross-Validation K=5", color="cyan")
    plt.errorbar(degrees10, MSE_CrossVal_K10, yerr=MSE_CrossVal_std_K10, fmt="o-", label="MSE Cross-Validation K=10", color="blue")
    plt.plot(bootstrap_degrees, bootstrap_error, "x-", label="Error/MSE Bootstrap", color="orangered")

    plt.yscale("log")
    plt.xlabel("Polynomial degree")
    plt.ylabel("MSE")
    plt.legend(loc="upper left", **legend_style)
    plt.tight_layout()
    plt.savefig("proj1_figures/OLS_CrossVal_Bootstrap_d).pdf", format="pdf")
    plt.show()


def plot_Ridge_CrossVal_combined(ridge_result_K5, ridge_result_K10):

    degrees, lambdas, MSE_CrossVal_K5, _         = ridge_result_K5
    degrees_K10, lambdas_K10, MSE_CrossVal_K10, _ = ridge_result_K10

    assert np.array_equal(degrees, degrees_K10), "polynomial degrees for k=5 and k=10 must be the same"
    assert np.array_equal(lambdas, lambdas_K10), "lambda parameters for k=5 and k=10 must be the same"

    mask         = lambdas <= 1e2      # larger lambdas only shrink everything to the mean
    lambdas_plot = lambdas[mask]
    log_lambda   = np.log10(lambdas_plot)

    MSE_plots    = (MSE_CrossVal_K5[:, mask], MSE_CrossVal_K10[:, mask])
    logMSE_plots = [np.log10(MSE_plot) for MSE_plot in MSE_plots]

    # same colour scale for both panels
    all_logMSE = np.concatenate([logMSE.ravel() for logMSE in logMSE_plots])
    norm       = mcolors.Normalize(vmin=np.percentile(all_logMSE, 5), vmax=np.percentile(all_logMSE, 95))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True, constrained_layout=True)

    for ax, k, MSE_plot, logMSE_plot in zip(axes, (5, 10), MSE_plots, logMSE_plots):

        best_idx    = np.unravel_index(np.argmin(MSE_plot), MSE_plot.shape)
        best_degree = degrees[best_idx[0]]
        best_lambda = lambdas_plot[best_idx[1]]
        print(f"Ridge K={k}: best degree {best_degree}, lambda {best_lambda}, CV MSE {MSE_plot[best_idx]}")

        mesh = ax.pcolormesh(log_lambda, degrees, logMSE_plot, shading="auto", cmap="viridis", norm=norm)
        ax.scatter(np.log10(best_lambda), best_degree, color="red", edgecolor="black", s=80, zorder=3)

        ax.set_title(fr"$k={k}$")
        ax.set_xlabel(r"$\log_{10}(\lambda)$")
        ax.text(0.04, 0.95, fr"$p={best_degree}$" + "\n" + fr"$\lambda={sci_notation(best_lambda)}$",
                transform=ax.transAxes, va="top", bbox=dict(facecolor="lightgray", edgecolor="gray", alpha=0.75))

    axes[0].set_ylabel("Polynomial degree")

    cbar = fig.colorbar(mesh, ax=axes, location="right", pad=0.03)
    cbar.set_label(r"$\log_{10}(\mathrm{MSE}_{CV})$")

    fig.savefig("proj1_figures/Ridge_CrossVal_K5_K10_d).pdf", format="pdf")
    plt.show()


plot_OLS_CrossVal(OLS_CrossValK5_result, OLS_CrossValK10_result, bootstrap_result_n100)
plot_Ridge_CrossVal_combined(Ridge_CrossValK5_result, Ridge_CrossValK10_result)

In [ ]:
"""
Setup for parts e)-i): the data from the top of the notebook (uniform x, seed 2026), 80/20 split,
features standardised with the training mean and std, centred y
"""
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2026)


# exercises/week35.py, cell 2, with pinv instead of inv
def OLS_parameters(X, y):
    return np.linalg.pinv(X.T @ X) @ X.T @ y


def Ridge_parameters(X, y, lam):
    """Minimises (1/n)||y - X theta||^2 + lam ||theta||^2"""
    n_rows, p = X.shape
    return np.linalg.solve(X.T @ X + n_rows * lam * np.eye(p), X.T @ y)


def scale_features(x_train, x_test, degree):
    # column-major, as in the runs behind the report: Adam's round-off bursts (part f) depend on the memory layout
    X_train = np.asfortranarray(design_matrix(x_train, degree))
    X_test  = np.asfortranarray(design_matrix(x_test, degree))

    X_mean = X_train.mean(axis=0)
    X_std  = X_train.std(axis=0)

    return (X_train - X_mean) / X_std, (X_test - X_mean) / X_std


def test_MSE(fit, x_train, x_test, y_train, y_test, degree):
    """fit(X, y) gives theta; fitted on centred y, so the intercept mean(y_train) is not penalised"""
    X_train, X_test = scale_features(x_train, x_test, degree)
    y_mean          = y_train.mean()
    theta           = fit(X_train, y_train - y_mean)
    return MSE(y_test, X_test @ theta + y_mean)

### Part e): Writing your own gradient descent code, with analytical gradients and automatic differentiation

Replace now the analytical expressions for the optimal parameters
$\boldsymbol{\theta}$ of OLS and Ridge regression with your own gradient descent
code. In this part we focus only on the simplest gradient descent approach with
a fixed learning rate $\eta$ (see the lectures and exercises of week 37 and
Sections 4.3–4.5 of the lecture notes),

$$
\boldsymbol{\theta}_{k+1} = \boldsymbol{\theta}_k - \eta\,\nabla_{\boldsymbol{\theta}}C(\boldsymbol{\theta}_k).
$$

Compute the gradient of the cost function in two independent ways:

1. **Analytically**, from the expressions you derived in the week-35 exercises,
   that is $\nabla_{\boldsymbol{\theta}}C = \frac{2}{n}\boldsymbol{X}^T(\boldsymbol{X}\boldsymbol{\theta}-\boldsymbol{y})$
   for OLS and the corresponding expression with the extra term
   $2\lambda\boldsymbol{\theta}$ for Ridge (check your normalization
   conventions).
2. By **automatic differentiation** (Section 4.14 of the lecture notes), using
   for example **JAX** (`jax.grad`) or **Autograd**. Write the cost function as
   an ordinary Python function of $\boldsymbol{\theta}$ and let the library
   produce its gradient.

Verify that the two gradients agree to machine precision for OLS and Ridge, and
use both in your gradient descent code. Discuss briefly why automatic
differentiation is neither symbolic nor numerical (finite-difference)
differentiation, and what its cost is relative to one evaluation of the cost
function.

Study and compare your results from parts a) and b) with your gradient descent
approach: do you converge to the closed-form solutions, and how many iterations
does it take? Discuss in particular the role of the learning rate, and relate
the largest learning rate for which the iteration converges to the largest
eigenvalue of the Hessian matrix $\frac{2}{n}\boldsymbol{X}^T\boldsymbol{X}$
(plus $2\lambda\boldsymbol{I}$ for Ridge), see Section 4.5 of the lecture notes.

The cell below shows the minimal version of the automatic-differentiation check
with JAX (install with `pip install jax jaxlib`). Note the 64-bit switch: JAX
defaults to 32-bit floats, which would hide the "machine precision" agreement.

In [ ]:
"""
Part e): analytical gradients and automatic differentiation, from the project starter notebook, cell 15
"""
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

degree, lmbda = 5, 1e-2
X, _          = scale_features(x_train, x_test, degree)
y_gd          = y_train - y_train.mean()
theta0        = rng.normal(size=X.shape[1])


def cost_ols(theta, X, y_gd):
    return jnp.mean((y_gd - X @ theta)**2)


def cost_ridge(theta, X, y_gd, lmbda):
    return jnp.mean((y_gd - X @ theta)**2) + lmbda * jnp.sum(theta**2)


grad_ols_ad   = jax.grad(cost_ols)
grad_ridge_ad = jax.grad(cost_ridge)

grad_ols_analytic   = 2.0 / len(y_gd) * X.T @ (X @ theta0 - y_gd)
grad_ridge_analytic = grad_ols_analytic + 2.0 * lmbda * theta0

print(f"max |AD - analytic|: OLS {np.max(np.abs(grad_ols_ad(theta0, X, y_gd) - grad_ols_analytic))}, "
      f"Ridge {np.max(np.abs(grad_ridge_ad(theta0, X, y_gd, lmbda) - grad_ridge_analytic))}")

# largest stable learning rate, eta < 2 / lambda_max(H)
hessian = 2.0 / len(y_gd) * X.T @ X
eta_max = 2.0 / np.linalg.eigvalsh(hessian).max()
print(f"eta_max (OLS) = {eta_max}")

In [ ]:
"""
Plain gradient descent with both gradients, compared with the closed-form solutions
"""
learning_rate         = 0.9 * eta_max
num_iters, tol        = 1000, 1e-8
n_samples, n_features = X.shape

theta_OLS_analytic   = np.zeros(n_features)
theta_Ridge_analytic = np.zeros(n_features)
theta_OLS_AD         = np.zeros(n_features)
theta_Ridge_AD       = np.zeros(n_features)

for t in range(num_iters):
    gradient_OLS   = 2 / n_samples * X.T @ (X @ theta_OLS_analytic - y_gd)
    gradient_Ridge = 2 / n_samples * X.T @ (X @ theta_Ridge_analytic - y_gd) + 2 * lmbda * theta_Ridge_analytic

    theta_OLS_analytic   = theta_OLS_analytic - learning_rate * gradient_OLS
    theta_Ridge_analytic = theta_Ridge_analytic - learning_rate * gradient_Ridge
    theta_OLS_AD         = theta_OLS_AD - learning_rate * grad_ols_ad(theta_OLS_AD, X, y_gd)
    theta_Ridge_AD       = theta_Ridge_AD - learning_rate * grad_ridge_ad(theta_Ridge_AD, X, y_gd, lmbda)

    if np.linalg.norm(gradient_OLS) < tol and np.linalg.norm(gradient_Ridge) < tol:
        print(f"converged at iteration {t}")
        break

theta_ols_cf   = OLS_parameters(X, y_gd)
theta_ridge_cf = Ridge_parameters(X, y_gd, lmbda)

print(f"|GD - closed form|: OLS {np.linalg.norm(theta_OLS_analytic - theta_ols_cf)}, "
      f"Ridge {np.linalg.norm(theta_Ridge_analytic - theta_ridge_cf)}")
print(f"max |analytic - AD| in theta: OLS {np.max(np.abs(theta_OLS_analytic - theta_OLS_AD))}, "
      f"Ridge {np.max(np.abs(theta_Ridge_analytic - theta_Ridge_AD))}")

In [ ]:
"""
Learning-rate scan: iterations to reach the closed form, eta as a fraction of eta_max = 2/lambda_max(H)
"""
def iterations_to_converge(theta_cf, lam, eta, max_iters=100000, tol=1e-6):
    """Plain GD from theta = 0 until |theta - theta_cf| < tol; nan if it diverges"""
    theta = np.zeros(n_features)
    for t in range(1, max_iters + 1):
        theta    = theta - eta * (2 / n_samples * X.T @ (X @ theta - y_gd) + 2 * lam * theta)
        distance = np.linalg.norm(theta - theta_cf)
        if distance < tol or distance > 1e6:
            break
    return t if distance < tol else np.nan


H_ols     = 2.0 / n_samples * X.T @ X
H_ridge   = H_ols + 2 * lmbda * np.eye(n_features)
fractions = [0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 1.01, 1.1]

iters_needed = {}
for name, H, theta_cf, lam in (("OLS", H_ols, theta_ols_cf, 0.0), ("Ridge", H_ridge, theta_ridge_cf, lmbda)):
    eigenvalues = np.linalg.eigvalsh(H)
    eta_max_H   = 2.0 / eigenvalues.max()

    iters_needed[name] = [iterations_to_converge(theta_cf, lam, frac * eta_max_H) for frac in fractions]
    print(f"{name}: eta_max {eta_max_H}, kappa {eigenvalues.max() / eigenvalues.min()}, "
          f"iterations {np.array(iters_needed[name])}")

fig, ax = plt.subplots()
# Role: used to find colors
ax.semilogy(fractions, iters_needed["OLS"], "o-", color="#777777", label="OLS")
ax.semilogy(fractions, iters_needed["Ridge"], "x-", color="#004488", label=rf"Ridge, $\lambda={lmbda}$")
ax.axvline(1.0, color="k", ls=":", label=r"$\eta_{\max}=2/\lambda_{\max}$")
ax.set_xlabel(r"$\eta/\eta_{\max}$")
ax.set_ylabel(r"iterations to $|\theta-\hat\theta|<10^{-6}$")
ax.set_title("plain gradient descent, degree 5")
ax.legend(**legend_style)
plt.tight_layout()
plt.show()

**Extra check for the report, part e):** iteration counts predicted from the Hessian spectrum, and the report figure.

In [ ]:
"""
Theory check, lecture notes Eq. (4.19): the slowest mode, max |1 - eta*lambda_i|, sets the iteration count
"""
iters_theory = {}
for name, H, theta_cf in (("OLS", H_ols, theta_ols_cf), ("Ridge", H_ridge, theta_ridge_cf)):
    eigenvalues, eigenvectors = np.linalg.eigh(H)
    initial_error             = eigenvectors.T @ (np.zeros(n_features) - theta_cf)
    eta_max_H                 = 2.0 / eigenvalues.max()

    iters_theory[name] = []
    for frac in fractions[:6]:
        factors = np.abs(1.0 - frac * eta_max_H * eigenvalues)
        slowest = np.argmax(factors)
        iters_theory[name].append(np.log(1e-6 / abs(initial_error[slowest])) / np.log(factors[slowest]))

    print(f"{name}: lambda in [{eigenvalues[0]}, {eigenvalues[-1]}], |e_0| {abs(initial_error[0])}, "
          f"measured {np.array(iters_needed[name][:6])}, predicted {np.round(iters_theory[name]).astype(int)}")


"""
Report figures (figures/*.pdf), legends outside the axes
"""
from matplotlib.lines import Line2D
os.makedirs("figures", exist_ok=True)

report_style = {"figure.figsize": (3.4, 2.4), "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
                "legend.fontsize": 6.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
                "lines.markersize": 4, "lines.linewidth": 1.1}
col   = (3.4, 2.4)
wide2 = (12, 5)


def legend_outside(fig, axes, where="below", ncol=2, extra=()):
    """One figure legend outside the axes; extra = more (handle, label) pairs"""
    handles, labels = [], []
    for ax in np.atleast_1d(axes):
        for handle, label in zip(*ax.get_legend_handles_labels()):
            if label not in labels:
                handles.append(handle)
                labels.append(label)
    for handle, label in extra:
        handles.append(handle)
        labels.append(label)

    if where == "below":
        anchor = dict(loc="upper center", bbox_to_anchor=(0.5, 0.0))
    else:
        anchor = dict(loc="lower center", bbox_to_anchor=(0.5, 1.0))
    fig.legend(handles, labels, ncol=ncol, **anchor, **legend_style)


def save_report_fig(fig, name, tight=True):
    if tight:
        fig.tight_layout()
    fig.savefig(f"figures/{name}.pdf", bbox_inches="tight")


with plt.rc_context(report_style):
    fig, ax = plt.subplots(figsize=col)
    for name, color, marker in (("OLS", "#777777", "o"), ("Ridge", "#004488", "x")):
        label = name if name == "OLS" else rf"Ridge, $\lambda={lmbda}$"
        ax.semilogy(fractions, iters_needed[name], marker, color=color, label=f"{label}, measured")
        ax.semilogy(fractions[:6], iters_theory[name], "-", color=color, label=f"{label}, slowest mode")
    ax.axvline(1.0, color="k", ls=":", label=r"$\eta_{\max}=2/\lambda_{\max}$")
    ax.set_xlabel(r"$\eta/\eta_{\max}$")
    ax.set_ylabel(r"iterations to $\|\theta-\hat\theta\|<10^{-6}$")
    legend_outside(fig, ax, ncol=2)
    save_report_fig(fig, "gd_eta")
    plt.show()

### Part f): Including momentum and more advanced ways to update the learning rate

We keep our focus on OLS and Ridge regression and update our gradient descent
code by including **momentum**, **AdaGrad**, **RMSprop** and **Adam** as methods
for iteratively updating the learning rate (Sections 4.6 and 4.8–4.11 of the
lecture notes; the lectures of weeks 37 and 38 contain several examples on how
to implement these methods). You are free to use either the analytical or the
automatically differentiated gradient here — the update rules do not care where
the gradient comes from.

Discuss the results and compare the different methods applied to the
one-dimensional Runge function, for example in terms of the number of iterations
needed to reach a given accuracy relative to the closed-form solution, and in
terms of their sensitivity to the choice of the (initial) learning rate.

<span style="color:lightcoral; font-size:23px">f)</span>

In [ ]:
"""
Part f): momentum, AdaGrad, RMSprop and Adam
"""
# runge_data: changed from WeeklyMaterial/week38/week38tuesday.ipynb to return the training data of parts e)-i)
def runge_data(degree=6):
    X_train, _ = scale_features(x_train, x_test, degree)
    return X_train, y_train - y_train.mean()


def closed_form(X, y, lam=0.0):
    return OLS_parameters(X, y) if lam == 0 else Ridge_parameters(X, y, lam)


# cost, gradient, hessian_eigs: WeeklyMaterial/week38/week38tuesday.ipynb, cell 3
def cost(theta, X, y, lam=0.0):
    """Eqs. (4.12) and (4.16)"""
    return np.sum((X @ theta - y)**2) / len(y) + lam * theta @ theta


def gradient(theta, X, y, lam=0.0):
    """Eqs. (4.13) and (4.17)"""
    return (2.0 / len(y)) * X.T @ (X @ theta - y) + 2.0 * lam * theta


def hessian_eigs(X, lam=0.0):
    """Eqs. (4.14) and (4.17)"""
    return np.linalg.eigvalsh((2.0 / len(X)) * X.T @ X + 2.0 * lam * np.eye(X.shape[1]))


# optimiser_step, optimise: WeeklyMaterial/week38/week38tuesday.ipynb, cell 7
def optimiser_step(method, theta, g, state, t, gamma, beta=0.9, rho=0.99, beta1=0.9, beta2=0.999, eps=1e-8):
    """One update from the gradient g at step t = 1, 2, ...; Eqs. (4.10), (4.28), (4.42)-(4.45), (4.47)-(4.48), (4.51)-(4.55)"""
    if method == "plain":
        return theta - gamma * g, state

    if method == "momentum":
        state["v"] = beta * state.get("v", 0.0) + gamma * g
        return theta - state["v"], state

    if method == "adagrad":
        state["r"] = state.get("r", 0.0) + g * g
        return theta - gamma * g / (np.sqrt(state["r"]) + eps), state

    if method == "rmsprop":
        state["r"] = rho * state.get("r", 0.0) + (1.0 - rho) * g * g
        return theta - gamma * g / (np.sqrt(state["r"]) + eps), state

    if method == "adam":
        state["m"] = beta1 * state.get("m", 0.0) + (1.0 - beta1) * g
        state["r"] = beta2 * state.get("r", 0.0) + (1.0 - beta2) * g * g
        m_hat      = state["m"] / (1.0 - beta1**t)
        r_hat      = state["r"] / (1.0 - beta2**t)
        return theta - gamma * m_hat / (np.sqrt(r_hat) + eps), state

    raise ValueError(f"unknown method {method}")


def optimise(grad, theta0, method, gamma, num_iters=1000, tol=0.0, **kw):
    """Full-gradient optimiser from theta0, returns all iterates"""
    theta, state = np.array(theta0, dtype=float), {}
    history      = [theta.copy()]
    for t in range(1, num_iters + 1):
        g            = grad(theta)
        theta, state = optimiser_step(method, theta, g, state, t, gamma, **kw)
        history.append(theta.copy())
        if np.linalg.norm(g) < tol:
            break
    return np.array(history)


# RUNGE_RUNS, runge_comparison: WeeklyMaterial/week38/week38tuesday.ipynb, cell 9, with lam added
# (method, learning rate (None: 0.9 * 2/lambda_max), colour, label)

#Tool: Open AI chatgpt
#Role: To get the color codes for these SICK colors
RUNGE_RUNS = (
    ("plain",    None, "#777777", "plain GD"),
    ("momentum", None, "#004488", r"momentum, $\beta=0.9$"),
    ("adagrad",  0.5,  "#DDAA33", r"AdaGrad, $\gamma=0.5$"),
    ("rmsprop",  0.01, "#228833", r"RMSProp, $\gamma=0.01$"),
    ("adam",     0.05, "#BB5566", r"Adam, $\gamma=0.05$"),
)


def runge_comparison(degree=10, num_iters=30000, lam=0.0):
    """Excess cost C(theta_k) - C(theta_hat) for the five optimisers"""
    X, y     = runge_data(degree=degree)
    c_min    = cost(closed_form(X, y, lam), X, y, lam)
    eigs     = hessian_eigs(X, lam)
    gamma_gd = 0.9 * 2.0 / eigs.max()

    excess = {}
    for method, gamma, _, _ in RUNGE_RUNS:
        path           = optimise(lambda th: gradient(th, X, y, lam), np.zeros(degree), method,
                                  gamma_gd if gamma is None else gamma, num_iters=num_iters)
        excess[method] = np.array([cost(theta, X, y, lam) - c_min for theta in path])
    return excess, eigs, gamma_gd


def first_below(excess, tol):
    """First iteration with excess cost below tol, None if never"""
    return int(np.argmax(excess < tol)) if (excess < tol).any() else None


lam_ridge = 1e-2
excess6, eigs6, gamma6        = runge_comparison(degree=6, num_iters=20000)
excess6_ridge, _, gamma6_ridge = runge_comparison(degree=6, num_iters=20000, lam=lam_ridge)

# print loop from WeeklyMaterial/week38/week38tuesday.ipynb, cell 10
for name, excess, gamma_gd in (("OLS", excess6, gamma6), ("Ridge", excess6_ridge, gamma6_ridge)):
    for method, gamma, _, _ in RUNGE_RUNS:
        e = excess[method]
        print(f"{name:5s} {method:8s} (gamma {gamma_gd if gamma is None else gamma}): "
              f"1e-4 at {first_below(e, 1e-4)}, 1e-8 at {first_below(e, 1e-8)}, final {max(e[-1], 0)}")

In [ ]:
fig, ax = plt.subplots()
for method, gamma, color, label in RUNGE_RUNS:
    ax.semilogy(np.maximum(excess6[method], 1e-16), color=color, label=label)
ax.set_xlabel("iteration")
ax.set_ylabel(r"$C(\theta_k) - C(\hat\theta)$")
ax.set_title("OLS convergence at degree 6")
ax.legend(**legend_style)
plt.tight_layout()
plt.show()

In [ ]:
# running minimum, monotonic decrease
fig, ax = plt.subplots()
for method, gamma, color, label in RUNGE_RUNS:
    ax.semilogy(np.maximum(np.minimum.accumulate(excess6[method]), 1e-16), color=color, label=label)
ax.set_xlabel("iteration")
ax.set_ylabel(r"$C(\theta_k) - C(\hat\theta)$")
ax.set_title("OLS convergence at degree 6")
ax.legend(**legend_style)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for method, gamma, color, label in RUNGE_RUNS:
    ax.semilogy(np.maximum(np.minimum.accumulate(excess6_ridge[method]), 1e-16), color=color, label=label)
ax.set_xscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel(r"$C(\theta_k) - C(\hat\theta)$")
ax.set_title(rf"Ridge convergence at degree 6, $\lambda={lam_ridge}$")
ax.legend(**legend_style)
plt.tight_layout()
plt.show()

In [ ]:
"""
Sensitivity to the learning rate: excess cost after a fixed number of iterations
"""
X6, y6     = runge_data(degree=6)
gamma_max6 = 2.0 / hessian_eigs(X6).max()
gammas     = np.logspace(-3, 1, 17)
n_it       = 2000


def final_excess_cost(method, learning_rate, lam, n_iterations=n_it):
    """C(theta) - C(theta_hat) after n_iterations, nan if the run diverged
    Tool: OpenAI chatgpt
    Role: had some issues with the previous implementation, so AI guided me to use the big for loop and the np.errstate to avoid overflow and invalid warnings
    """
    c_min = cost(closed_form(X6, y6, lam), X6, y6, lam)
    with np.errstate(over="ignore", invalid="ignore"):
        theta  = optimise(lambda th: gradient(th, X6, y6, lam), np.zeros(X6.shape[1]), method, learning_rate,
                          num_iters=n_iterations)[-1]
        excess = cost(theta, X6, y6, lam) - c_min
    return excess if np.isfinite(excess) and excess < 1e2 else np.nan


sens = {}
for lam, name, title in ((0.0, "OLS", "OLS"), (lam_ridge, "Ridge", rf"Ridge, $\lambda={lam_ridge}$")):
    fig, ax = plt.subplots()
    for (method, _, color, _), marker in zip(RUNGE_RUNS, markers):
        sens[(lam, method)] = np.maximum([final_excess_cost(method, g, lam) for g in gammas], 1e-16)
        ax.loglog(gammas, sens[(lam, method)], marker + "-", color=color, label=method)

        ok = gammas[sens[(lam, method)] < 1e-6]
        print(f"{name:5s} {method:8s}: excess < 1e-6 for gamma in " + (f"[{ok.min()}, {ok.max()}]" if len(ok) else "none"))

    ax.axvline(gamma_max6, color="k", ls=":", label=r"$2/\lambda_{\max}$")
    ax.set_xlabel(r"learning rate $\gamma$")
    ax.set_ylabel(rf"$C(\theta) - C(\hat\theta)$ after {n_it} iterations")
    ax.set_title(f"{title}, degree 6, missing points diverged")
    ax.legend(**legend_style)
    plt.tight_layout()
    plt.show()

**Extra checks for the report, part f):** momentum rate from the eigenvalues, the RMSprop/Adam floor versus the learning rate, and the report figures.

In [ ]:
"""
Theory check for plain GD and momentum, Eqs. (4.19) and (4.28): per eigenmode the error contracts by the largest
root r of z^2 - (1 + beta - eta*lambda_i) z + beta = 0, so 1e-4 -> 1e-8 in excess cost takes ln(1e4) / (-2 ln r) iterations
"""
for lam, excess in ((0.0, excess6), (lam_ridge, excess6_ridge)):
    eigenvalues = hessian_eigs(X6, lam)
    eta         = 0.9 * 2.0 / eigenvalues.max()
    print(f"lambda {lam}: kappa {eigenvalues.max() / eigenvalues.min()}, lambda_min {eigenvalues.min()}, eta {eta}")

    for method, beta in (("plain", 0.0), ("momentum", 0.9)):
        r         = max(np.max(np.abs(np.roots([1.0, -(1.0 + beta - eta * l), beta]))) for l in eigenvalues)
        predicted = np.log(1e4) / (-2 * np.log(r))
        measured  = first_below(excess[method], 1e-8) - first_below(excess[method], 1e-4)
        print(f"  {method:8s}: r {r}, 1e-4 -> 1e-8: predicted {predicted}, measured {measured}")


"""
RMSprop/Adam with a constant learning rate end in a limit cycle (Eq. (4.66)), the excess cost floors at ~eta^2
"""
def adaptive_floor(method, learning_rate, lam):
    """Mean excess cost over the last 4000 of 40000 iterations"""
    c_min = cost(closed_form(X6, y6, lam), X6, y6, lam)
    path  = optimise(lambda th: gradient(th, X6, y6, lam), np.zeros(6), method, learning_rate, num_iters=40000)
    return np.mean([cost(theta, X6, y6, lam) - c_min for theta in path[-4000:]])


gammas_floor = np.sort(np.append(np.logspace(-4, -1, 7), 0.05))     # 0.05 is the Adam rate used above
floor        = {}
for lam in (0.0, lam_ridge):
    for method in ("rmsprop", "adam"):
        floor[(lam, method)] = np.array([adaptive_floor(method, g, lam) for g in gammas_floor])
        slope                = np.polyfit(np.log(gammas_floor), np.log(floor[(lam, method)]), 1)[0]
        print(f"lambda {lam}, {method}: slope {slope}, floor/eta^2 {np.round(floor[(lam, method)] / gammas_floor**2, 2)}")
print(f"tr(H)/2 = {hessian_eigs(X6).sum() / 2}")

bursts = ", ".join(f"k={k}: {max(excess6_ridge['adam'][k], 0)}" for k in (248, 1000, 5000, 8000, 10000, 15000, 20000))
print(f"Adam, Ridge: {bursts}")


"""
Report figures
"""
with plt.rc_context(report_style):
    fig, axes = plt.subplots(1, 2, figsize=wide2, sharey=True)
    for ax, excess, title in ((axes[0], excess6, "OLS"), (axes[1], excess6_ridge, rf"Ridge, $\lambda={lam_ridge}$")):
        for method, gamma, color, label in RUNGE_RUNS:
            label = label.replace(r"\gamma", r"\eta") if gamma else label + rf", $\eta={gamma6}$"
            ax.loglog(np.arange(1, len(excess[method]) + 1), np.maximum(excess[method], 1e-16), color=color, label=label,
                      zorder=1 if method == "adam" else 2)
        ax.set_xlabel("iteration $k$")
        ax.set_title(f"{title}, degree 6")
    axes[0].set_ylabel(r"$C(\theta_k)-C(\hat\theta)$")
    legend_outside(fig, axes, where="above", ncol=5)
    save_report_fig(fig, "optimizers")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=wide2, sharey=True)
    for ax, lam, title in ((axes[0], 0.0, "OLS"), (axes[1], lam_ridge, rf"Ridge, $\lambda={lam_ridge}$")):
        for (method, _, color, label), marker in zip(RUNGE_RUNS, markers):
            ax.loglog(gammas, sens[(lam, method)], marker + "-", color=color, label=label.split(",")[0])
        ax.axvline(gamma_max6, color="k", ls=":", label=r"$2/\lambda_{\max}$")
        ax.set_xlabel(r"learning rate $\eta$")
        ax.set_title(f"{title}, degree 6, after {n_it} iterations")
    axes[0].set_ylabel(r"$C(\theta)-C(\hat\theta)$")
    legend_outside(fig, axes, where="above", ncol=6)
    save_report_fig(fig, "lr_sensitivity")
    plt.show()

    fig, ax = plt.subplots(figsize=col)
    # Role: used again to find these cool colors for the plot
    for (lam, method), color, marker in zip(floor, ("#228833", "#BB5566", "#228833", "#BB5566"), ("o", "^", "x", "v")):
        label = ("RMSprop" if method == "rmsprop" else "Adam") + (", OLS" if lam == 0 else ", Ridge")
        ax.loglog(gammas_floor, floor[(lam, method)], marker, ls="-" if lam == 0 else "--", color=color, label=label)
    ax.loglog(gammas_floor, 4.6 * gammas_floor**2, color="k", lw=0.8, ls=":", label=r"$4.6\,\eta^2$")
    ax.set_xlabel(r"learning rate $\eta$")
    ax.set_ylabel("mean excess cost, last 4000 it.")
    legend_outside(fig, ax, ncol=3)
    save_report_fig(fig, "adaptive_floor")
    plt.show()

### Part g): Writing our own code for Lasso regression

Lasso regression (Section 3.9 of the lecture notes, and the lectures of weeks 35
and 36) represents our first encounter with a machine learning method which
cannot be solved through analytical expressions (as in OLS and Ridge
regression). Use the gradient descent methods you developed in parts e) and f)
to solve the Lasso optimization problem.

The $\ell_1$ penalty $\lambda\|\boldsymbol{\theta}\|_1$ is not differentiable
at $\theta_j=0$. Discuss how you could handle this: what does your automatic
differentiation tool return for the derivative of $|\theta|$ at zero, and is
that a valid subgradient? Based on the gradient, write your own code for Lasso
regression.

You can compare your results with the functionalities of **Scikit-Learn**; be
careful with the different conventions for the penalty parameter (`alpha`) and
the normalization of the cost function discussed in the week-35 and week-36
Tuesday sessions.

Discuss (critically) your results for the Runge function from OLS, Ridge and
Lasso regression using the various gradient descent approaches.

In [ ]:
"""
Part g): Lasso. First three prints from the project starter notebook, cell 20
"""
print(f"jax.grad(|theta|): at 0 {jax.grad(jnp.abs)(0.0)}, at -0.3 {jax.grad(jnp.abs)(-0.3)}, at 0.3 {jax.grad(jnp.abs)(0.3)}; "
      f"np.sign(0) = {np.sign(0.0)}")

from sklearn.linear_model import Lasso


# cost_lasso, gradient_lasso: adapted from WeeklyMaterial/week38/week38tuesday.ipynb, cell 3
def cost_lasso(theta, X, y, lam):
    return np.sum((X @ theta - y)**2) / len(y) + lam * np.sum(np.abs(theta))


def gradient_lasso(theta, X, y, lam):
    """Subgradient with sign(0) = 0"""
    return (2.0 / len(y)) * X.T @ (X @ theta - y) + lam * np.sign(theta)


# sklearn minimises (1/(2n))||y - X theta||^2 + alpha ||theta||_1, so alpha = lambda/2 (week35tuesday.ipynb, cell 17)
def lasso_skl(X, y, lam):
    return Lasso(alpha=lam / 2.0, fit_intercept=False, max_iter=100000, tol=1e-12).fit(X, y).coef_


# adapted from ridge_cost_jax in WeeklyMaterial/week37/week37tuesday.ipynb, cell 17
def cost_lasso_jax(theta, X, y, lam):
    return jnp.mean((y - X @ theta)**2) + lam * jnp.sum(jnp.abs(theta))


Xr, yr     = runge_data(degree=6)
lam_lasso  = 1e-2
theta_test = rng.normal(size=Xr.shape[1])
ad_error   = np.max(np.abs(jax.grad(cost_lasso_jax)(theta_test, Xr, yr, lam_lasso) - gradient_lasso(theta_test, Xr, yr, lam_lasso)))
print(f"Lasso max |AD - analytic| = {ad_error}")

theta_skl = lasso_skl(Xr, yr, lam_lasso)
gamma_gd  = 0.9 * 2.0 / hessian_eigs(Xr).max()
print(f"sklearn: {np.round(theta_skl, 5)}")


"""
Subgradient descent with all methods from part f), adapted from runge_comparison (week38tuesday.ipynb, cells 9 and 10)
"""
c_min        = cost_lasso(theta_skl, Xr, yr, lam_lasso)
excess_lasso = {}
for method, gamma, _, _ in RUNGE_RUNS:
    learning_rate        = gamma_gd if gamma is None else gamma
    path                 = optimise(lambda th: gradient_lasso(th, Xr, yr, lam_lasso), np.zeros(Xr.shape[1]), method,
                                    learning_rate, num_iters=20000)
    excess_lasso[method] = np.array([cost_lasso(theta, Xr, yr, lam_lasso) - c_min for theta in path])

    e = excess_lasso[method]
    print(f"{method} (gamma {learning_rate}): 1e-4 at {first_below(e, 1e-4)}, 1e-8 at {first_below(e, 1e-8)}, "
          f"final {max(e[-1], 0)}")

In [ ]:
fig, ax = plt.subplots()
for method, gamma, color, label in RUNGE_RUNS:
    ax.semilogy(np.maximum(np.minimum.accumulate(excess_lasso[method]), 1e-16), color=color, label=label)
ax.set_xscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel(r"$C(\theta_k) - C(\hat\theta)$")
ax.set_title(rf"Lasso convergence at degree 6, $\lambda={lam_lasso}$")
ax.legend(**legend_style)
plt.tight_layout()
plt.show()


"""
Ridge and Lasso parameters against lambda, inspired by WeeklyMaterial/week35/week35tuesday.ipynb, cell 21
"""
lambdas_path = np.logspace(-5, 0, 40)
ridge_path   = np.array([closed_form(Xr, yr, lam) for lam in lambdas_path])
lasso_path   = np.array([lasso_skl(Xr, yr, lam) for lam in lambdas_path])

for name, path in (("Ridge", ridge_path), ("Lasso", lasso_path)):
    fig, ax = plt.subplots()
    for j in range(Xr.shape[1]):
        ax.semilogx(lambdas_path, path[:, j], label=rf"$x^{j+1}$")
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_xlabel(r"$\lambda$"); ax.set_ylabel(r"$\theta_j$")
    ax.set_title(f"{name} parameters, degree 6")
    ax.legend(**legend_style)
    plt.tight_layout()
    plt.show()

**Extra checks for the report, part g):** the subgradient floor against the learning rate, the AD subgradient, and the report figures.

In [ ]:
"""
Constant-step subgradient descent: the floor shrinks with eta, but exact zeros never appear
"""
c_min = cost_lasso(theta_skl, Xr, yr, lam_lasso)

for learning_rate in (0.292, 0.1, 0.03, 0.01):
    path       = optimise(lambda th: gradient_lasso(th, Xr, yr, lam_lasso), np.zeros(Xr.shape[1]), "plain",
                          learning_rate, num_iters=40000)
    floor_mean = np.mean([cost_lasso(theta, Xr, yr, lam_lasso) - c_min for theta in path[-4000:]])
    print(f"eta {learning_rate}: floor {floor_mean}, max |theta_j| where sklearn = 0: "
          f"{np.abs(path[-1][theta_skl == 0]).max()}, exact zeros {np.sum(path[-1] == 0)}")


"""
The AD subgradient (d|theta|/dtheta = 1 at 0) used directly in plain GD
"""
grad_lasso_ad = jax.jit(jax.grad(cost_lasso_jax))
theta         = np.zeros(Xr.shape[1])
difference    = np.array(grad_lasso_ad(theta, Xr, yr, lam_lasso)) - gradient_lasso(theta, Xr, yr, lam_lasso)
print(f"AD - sign(0)=0 gradient at 0: {np.round(difference, 5)}")

for t in range(20000):
    theta = theta - gamma_gd * np.array(grad_lasso_ad(theta, Xr, yr, lam_lasso))
print(f"plain GD, AD subgradient: excess {cost_lasso(theta, Xr, yr, lam_lasso) - c_min}")


"""
Report figures
"""
with plt.rc_context(report_style):
    fig, ax = plt.subplots(figsize=col)

    # raw curve up to k = 100, then a running mean over 100 iterations
    window = 100
    for method, gamma, color, label in RUNGE_RUNS:
        label      = label.replace(r"\gamma", r"\eta") if gamma else label
        e          = np.maximum(excess_lasso[method], 1e-16)
        smooth     = np.convolve(e, np.ones(window) / window, mode="valid")
        iterations = np.arange(1, len(e) + 1)
        ax.loglog(iterations, np.concatenate([e[:window], smooth[1:]]), color=color, label=label)

    ax.set_xlabel("iteration $k$")
    ax.set_ylabel(r"$C(\theta_k)-C(\hat\theta)$")
    ax.set_title(rf"Lasso, degree 6, $\lambda={lam_lasso}$")
    legend_outside(fig, ax, ncol=2)
    save_report_fig(fig, "lasso_conv")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=wide2, sharey=True)
    for ax, path, name in ((axes[0], ridge_path, "Ridge"), (axes[1], lasso_path, "Lasso")):
        for j in range(Xr.shape[1]):
            ax.semilogx(lambdas_path, path[:, j], label=rf"$\theta_{j + 1}$ ($x^{j + 1}$)")
        ax.axhline(0, color="gray", lw=0.5)
        ax.set_xlabel(r"$\lambda$")
        ax.set_title(f"{name} parameters, degree 6")
    axes[0].set_ylabel(r"$\theta_j$")
    legend_outside(fig, axes, where="above", ncol=6)
    save_report_fig(fig, "lasso_paths")
    plt.show()

### Part h): Stochastic gradient descent

Our last gradient step is to include stochastic gradient descent (Section 4.7
of the lecture notes) using the same methods to update the learning rates as in
parts e)–g). Split the training data into mini-batches, loop over epochs, and
study the dependence on the mini-batch size, the number of epochs and the
learning-rate schedule. Compare and discuss your results with and without
stochastic gradient descent and give a critical assessment of the various
methods — both in terms of the final accuracy relative to the closed-form
solutions of parts a) and b), and in terms of computational cost.

In [ ]:
"""
Part h): stochastic gradient descent. make_batches: WeeklyMaterial/week39/week39monday.ipynb, cell 6,
train: adapted from the same cell
"""
import time


def make_batches(n, M, rng):
    """Shuffle once per epoch and split into minibatches of size M"""
    idx = rng.permutation(n)
    return [idx[k:k + M] for k in range(0, n, M)]


def train(grad, X, y, method="plain", gamma=0.1, M=None, n_epochs=10, t0=None, seed=2026):
    """Minibatch SGD with the optimisers from part f), M = None is full-batch GD.
    gamma_t = gamma * t0 / (t + t0) if t0 is given. Returns theta after every epoch"""
    n, p = X.shape
    M    = n if M is None else M
    rng  = np.random.default_rng(seed)

    theta, state, t = np.zeros(p), {}, 0
    history         = [theta.copy()]
    for epoch in range(n_epochs):
        for batch in make_batches(n, M, rng):
            t           += 1
            gamma_t      = gamma if t0 is None else gamma * t0 / (t + t0)
            theta, state = optimiser_step(method, theta, grad(theta, X[batch], y[batch]), state, t, gamma_t)
        history.append(theta.copy())
    return np.array(history)


Xr, yr               = runge_data(degree=6)
lam_ridge, lam_lasso = 1e-2, 1e-2
n_epochs             = 500


"""
Dependence on the minibatch size and the learning-rate schedule, plain SGD with gamma = 0.1
"""
batch_sizes = [5, 10, 20, 50, None]
excess_M    = {}
for lam in (0.0, lam_ridge):
    c_min = cost(closed_form(Xr, yr, lam), Xr, yr, lam)
    for M in batch_sizes:
        for t0 in (None, 1000):
            history = train(lambda th, X, y: gradient(th, X, y, lam), Xr, yr, "plain", 0.1, M=M, n_epochs=n_epochs, t0=t0)
            e       = np.array([cost(theta, Xr, yr, lam) - c_min for theta in history])
            excess_M[(lam, M, t0)] = e
            print(f"lambda {lam}, M {str(M):4s}, t0 {str(t0):4s}: epoch 10/100/{n_epochs}: {e[10]} / {e[100]} / {e[-1]}")


"""
All methods, GD against SGD (M = 10), for OLS, Ridge and Lasso.
Momentum needs a smaller rate with minibatches, the effective step is gamma/(1 - beta)
"""
gammas_gd  = {"plain": 0.3, "momentum": 0.3,  "adagrad": 0.5, "rmsprop": 0.01, "adam": 0.05}
gammas_sgd = {"plain": 0.1, "momentum": 0.02, "adagrad": 0.5, "rmsprop": 0.01, "adam": 0.05}
M_sgd      = 10

# name: (minibatch gradient, full cost, reference solution)
problems = {
    "OLS":   (lambda th, X, y: gradient(th, X, y),
              lambda th: cost(th, Xr, yr),
              closed_form(Xr, yr)),
    "Ridge": (lambda th, X, y: gradient(th, X, y, lam_ridge),
              lambda th: cost(th, Xr, yr, lam_ridge),
              closed_form(Xr, yr, lam_ridge)),
    "Lasso": (lambda th, X, y: gradient_lasso(th, X, y, lam_lasso),
              lambda th: cost_lasso(th, Xr, yr, lam_lasso),
              lasso_skl(Xr, yr, lam_lasso)),
}

for name, (grad_batch, cost_full, theta_ref) in problems.items():
    c_ref = cost_full(theta_ref)
    for method in gammas_gd:
        for M, label, gamma in ((None, "GD ", gammas_gd[method]), (M_sgd, "SGD", gammas_sgd[method])):
            t_start   = time.perf_counter()
            history   = train(grad_batch, Xr, yr, method, gamma, M=M, n_epochs=n_epochs)
            time_used = time.perf_counter() - t_start

            excess   = cost_full(history[-1]) - c_ref
            distance = np.linalg.norm(history[-1] - theta_ref)
            print(f"{name:5s} {label} {method:8s} (gamma {gamma}): excess {max(excess, 0)}, "
                  f"|theta - ref| {distance}, {time_used * 1e3} ms")

In [ ]:
for lam in (0.0, lam_ridge):
    fig, ax = plt.subplots()
    # Role: used again to find these cool colors for the plot
    for M, color in ((5, "#BB5566"), (20, "#228833"), (None, "#777777")):
        label = "GD (M = n)" if M is None else f"M = {M}"
        ax.semilogy(np.maximum(excess_M[(lam, M, None)], 1e-16), color=color, label=label)
        ax.semilogy(np.maximum(excess_M[(lam, M, 1000)], 1e-16), color=color, ls="--")
    ax.set_xlabel("epoch"); ax.set_ylabel(r"$C(\theta) - C(\hat\theta)$")
    ax.set_title("OLS" if lam == 0 else rf"Ridge, $\lambda={lam}$")
    ax.legend(title=r"solid: $\gamma=0.1$, dashed: decaying", **legend_style)
    plt.tight_layout()
    plt.show()

**Extra checks for the report, part h):** the SGD noise floor against Eq. (4.51) of the lecture notes, minibatch curvature, the cost of the closed form, and the report figure.

In [ ]:
lam      = lam_ridge
theta_r  = closed_form(Xr, yr, lam)
c_r      = cost(theta_r, Xr, yr, lam)
n_tr     = len(yr)

per_example_gradients = 2 * Xr * (Xr @ theta_r - yr)[:, None] + 2 * lam * theta_r
trS                   = np.mean(np.sum(per_example_gradients**2, axis=1))


def sgd_floor(eta, M, replace, n_epochs=4000, seed=0):
    rng_sgd     = np.random.default_rng(seed)
    theta       = theta_r.copy()
    excess_cost = []
    for epoch in range(n_epochs):
        if replace:
            batches = [rng_sgd.choice(n_tr, M) for _ in range(n_tr // M)]
        else:
            batches = make_batches(n_tr, M, rng_sgd)
        for batch in batches:
            theta = theta - eta * gradient(theta, Xr[batch], yr[batch], lam)
        if epoch >= 1000:
            excess_cost.append(cost(theta, Xr, yr, lam) - c_r)
    return np.mean(excess_cost)


floor_rows = []
for eta in (0.025, 0.05, 0.1):
    for M in (5, 10, 20):
        predicted        = eta * trS / (4 * M)
        with_replacement = sgd_floor(eta, M, replace=True)
        reshuffled       = sgd_floor(eta, M, replace=False)
        floor_rows.append((eta, M, predicted, with_replacement, reshuffled))
        print(f"eta {eta}, M {M}: measured/predicted: replacement {with_replacement / predicted}, "
              f"reshuffled {reshuffled / predicted}")
floor_rows = np.array(floor_rows)
print(f"tr S = {trS}")


"""
Largest curvature seen by one minibatch: a step is stable only if eta < 2/lambda_max of the minibatch Hessian
"""
rng_curvature = np.random.default_rng(1)
for M in (5, 10, 20, n_tr):
    batches = [rng_curvature.choice(n_tr, M, replace=False) for _ in range(2000)]
    lmax    = max(hessian_eigs(Xr[batch], lam).max() for batch in batches)
    print(f"M {M}: max minibatch lambda_max {lmax}, 2/lambda_max {2 / lmax}")

t_start = time.perf_counter()
for _ in range(1000):
    closed_form(Xr, yr, lam)
print(f"closed-form Ridge: {(time.perf_counter() - t_start)} ms per solve")


"""
Report figures
"""
with plt.rc_context(report_style):
    fig, axes = plt.subplots(1, 2, figsize=wide2, sharey=True)
    for ax, lam_, title in ((axes[0], 0.0, "OLS"), (axes[1], lam_ridge, rf"Ridge, $\lambda={lam_ridge}$")):
        # Role: used again to find these cool colors for the plot
        for M, color in ((5, "#BB5566"), (20, "#228833"), (None, "k")):
            label = "GD ($M=n$)" if M is None else f"$M={M}$"
            for t0, ls in ((None, "-"), (1000, "--")):
                e      = np.maximum(excess_M[(lam_, M, t0)], 1e-16)
                smooth = np.exp(np.convolve(np.log(e), np.ones(20) / 20, mode="valid"))
                ax.semilogy(e, color=color, ls=ls, lw=0.4, alpha=0.25)
                ax.semilogy(np.arange(19, len(e)), smooth, color=color, ls=ls, lw=1.3, label=label if t0 is None else None)
        ax.set_xlabel("epoch")
        ax.set_title(f"plain SGD, {title}")
    axes[0].set_ylabel(r"$C(\theta)-C(\hat\theta)$")
    legend_outside(fig, axes, where="above", ncol=5,
                   extra=((Line2D([], [], color="k", lw=1.0), r"constant $\eta=0.1$"),
                          (Line2D([], [], color="k", lw=1.0, ls="--"), "decaying, Eq. (4.52)")))
    save_report_fig(fig, "sgd_batch")
    plt.show()

    fig, ax = plt.subplots(figsize=col)
    for M, marker in ((5, "o"), (10, "x"), (20, "^")):
        rows = floor_rows[:, 1] == M
        ax.loglog(floor_rows[rows, 2], floor_rows[rows, 3], marker, color="#004488", label=f"with replacement, $M={M}$")
        ax.loglog(floor_rows[rows, 2], floor_rows[rows, 4], marker, color="#BB5566", label=f"reshuffled, $M={M}$")
    limits = np.array([3e-5, 5e-2])
    ax.loglog(limits, limits, "k:", lw=0.8, label="Eq. (4.51) of the lecture notes")
    ax.set_ylim(bottom=3e-6)
    ax.set_xlabel(r"predicted floor $\eta\,\mathrm{tr}\,S/(4M)$")
    ax.set_ylabel("measured mean excess cost")
    ax.set_title(r"plain SGD noise floor, Ridge, $\lambda=0.01$")
    legend_outside(fig, ax, ncol=2)
    save_report_fig(fig, "sgd_floor")
    plt.show()

### Part i): Final analysis — OLS, Ridge and Lasso with cross-validation

Bring the pieces together. Using the cross-validation setup from part d),
perform a final model selection for the Runge function with all three methods: OLS as a function of the polynomial
degree, and Ridge and Lasso as functions of both the polynomial degree and
$\lambda$. Present the results as a small number of well-chosen figures or
tables, identify the best model according to the cross-validated test error, and
discuss critically the strengths and weaknesses of the three methods for this
problem. Which of the terms in the bias-variance decomposition of part c) does
each method attack, and does your numerical evidence support that reading?

In [ ]:
"""
Part i): OLS, Ridge and Lasso with k-fold cross-validation on the data of parts e)-h)
"""
# Tool: OpenAI chatgpt
# Role: used to ignore convergence warnings from sklearn's Lasso when lambda is too small
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

polynomial_degrees = np.arange(1, 16)
lambda_parameters  = np.logspace(-10, 0, 21)


def ols_fit(X, y, lam):
    return OLS_parameters(X, y)


def ridge_fit(X, y, lam):
    return Ridge_parameters(X, y, lam)


# as lasso_fit in LectureNotes/chapter3.ipynb, cell 57; alpha = lambda/2 as in part g)
def lasso_fit(X, y, lam):
    """nan if the solver has not converged (small lambda at high degree), those points are left out"""
    model = Lasso(alpha=lam / 2.0, fit_intercept=False, max_iter=100000, tol=1e-8).fit(X, y)
    return model.coef_ if model.n_iter_ < model.max_iter else np.full(X.shape[1], np.nan)


# adapted from cv_curve in LectureNotes/chapter3.ipynb, cell 57
def cv_folds(fit, x, y, degree, lam=0.0, k=10, seed=2026):
    """Test MSE of every fold; scaling and centring learnt from the training folds only"""
    kfold = KFold(n_splits=k, shuffle=True, random_state=seed)
    return np.array([test_MSE(lambda X_, y_: fit(X_, y_, lam), x[train_idx], x[test_idx], y[train_idx], y[test_idx], degree)
                     for train_idx, test_idx in kfold.split(x)])


def cv_mse(fit, x, y, degree, lam=0.0, k=5, seed=2026):
    return np.mean(cv_folds(fit, x, y, degree, lam, k, seed))


def cv_grid(fit, k, lambdas=lambda_parameters, seed=2026):
    """CV MSE for every (degree, lambda)"""
    return np.array([[cv_mse(fit, x, y, degree, lam, k, seed) for lam in lambdas] for degree in polynomial_degrees])


results = {}
for k in (5, 10):
    mse_ols   = np.array([cv_mse(ols_fit, x, y, degree, k=k) for degree in polynomial_degrees])
    mse_ridge = cv_grid(ridge_fit, k)
    mse_lasso = cv_grid(lasso_fit, k)
    results[k] = (mse_ols, mse_ridge, mse_lasso)

    best = f"k = {k:2d}: OLS degree {polynomial_degrees[np.argmin(mse_ols)]} ({mse_ols.min()})"
    for name, mse in (("Ridge", mse_ridge), ("Lasso", mse_lasso)):
        i, j  = np.unravel_index(np.nanargmin(mse), mse.shape)
        best += f", {name} degree {polynomial_degrees[i]}, lambda {lambda_parameters[j]} ({mse[i, j]})"
    print(best + f"; Lasso not converged {np.isnan(mse_lasso).sum()}/{mse_lasso.size}")

# k = 10 from here on
mse_ols, mse_ridge, mse_lasso = results[10]

# nonzero parameters of the best Lasso model, fitted on all data
i, j       = np.unravel_index(np.nanargmin(mse_lasso), mse_lasso.shape)
X_all      = design_matrix(x, polynomial_degrees[i])
theta_best = lasso_fit((X_all - X_all.mean(axis=0)) / X_all.std(axis=0), y - y.mean(), lambda_parameters[j])
print(f"best Lasso theta ({np.sum(theta_best != 0)}/{len(theta_best)} nonzero): {np.round(theta_best, 4)}")

In [ ]:
fig, ax = plt.subplots()
for k, ls in ((10, "-"), (5, "--")):
    m_ols, m_ridge, m_lasso = results[k]
    ax.semilogy(polynomial_degrees, m_ols, "o" + ls, color="#777777", label=f"OLS, k = {k}")
    ax.semilogy(polynomial_degrees, m_ridge.min(axis=1), "x" + ls, color="#004488", label=rf"Ridge, best $\lambda$, k = {k}")
    ax.semilogy(polynomial_degrees, np.nanmin(m_lasso, axis=1), "^" + ls, color="#BB5566", label=rf"Lasso, best $\lambda$, k = {k}")
ax.axhline(sigma**2, color="k", ls=":", label=r"$\sigma^2$")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("CV MSE")
ax.set_title("cross-validated test error")
ax.legend(ncol=2, **legend_style)
plt.tight_layout()
plt.show()

for mse, name in ((mse_ridge, "Ridge"), (mse_lasso, "Lasso")):
    fig, ax = plt.subplots()
    im      = ax.pcolormesh(np.log10(lambda_parameters), polynomial_degrees, np.log10(mse), shading="nearest", cmap="viridis")
    i, j    = np.unravel_index(np.nanargmin(mse), mse.shape)
    ax.plot(np.log10(lambda_parameters[j]), polynomial_degrees[i], "r*", ms=12)
    ax.set_xlabel(r"$\log_{10}\lambda$"); ax.set_ylabel("polynomial degree")
    ax.set_title(f"{name}, 10-fold CV, $\\log_{{10}}$ MSE" + (" (white: not converged)" if name == "Lasso" else ""))
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

In [ ]:
"""
Which bias-variance term does each method attack: bootstrap with OLS, Ridge and Lasso fits on the same data
and bootstrap samples
"""
# adapted from bias_variance_sweep in WeeklyMaterial/week36/week36tuesday.ipynb, cell 6
def bias_variance(x_data, y_data, degrees, n_bootstraps=100, seed=2026, fit=OLS_parameters):
    """Bootstrap test error, bias^2 (with y), bias^2 (with f) and variance; fit(X, y) gives theta"""
    x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=seed)
    rng_bootstrap = np.random.default_rng(seed)

    error, bias2, bias2_f, variance = (np.zeros(len(degrees)) for _ in range(4))
    for k, degree in enumerate(degrees):
        y_pred = np.empty((len(y_test), n_bootstraps))
        for b in range(n_bootstraps):
            idx              = rng_bootstrap.integers(0, len(x_train), len(x_train))
            X_train, X_test  = scale_features(x_train[idx], x_test, degree)
            y_mean           = y_train[idx].mean()
            y_pred[:, b]     = X_test @ fit(X_train, y_train[idx] - y_mean) + y_mean

        y_pred_mean = y_pred.mean(axis=1)
        error[k]    = np.mean((y_test[:, None] - y_pred)**2)
        bias2[k]    = np.mean((y_test - y_pred_mean)**2)
        bias2_f[k]  = np.mean((runge_function(x_test) - y_pred_mean)**2)
        variance[k] = np.mean(np.var(y_pred, axis=1))
    return error, bias2, bias2_f, variance


ridge_with = lambda lam: (lambda X_, y_: Ridge_parameters(X_, y_, lam))
lasso_with = lambda lam: (lambda X_, y_: lasso_fit(X_, y_, lam))

# against degree, lambda = 1e-2 (the Lasso solver converges here for all degrees)
degrees_bv = np.arange(1, 16)
lam_bv     = 1e-2
bv_methods = {"OLS":   bias_variance(x, y, degrees_bv),
              "Ridge": bias_variance(x, y, degrees_bv, fit=ridge_with(lam_bv)),
              "Lasso": bias_variance(x, y, degrees_bv, fit=lasso_with(lam_bv))}

# against lambda, degree 12
deg_bv  = 12
lams_bv = np.logspace(-8, 0, 17)
ols_12  = np.ravel(bias_variance(x, y, [deg_bv]))
bv_lam  = {}
for name, fit_with in (("Ridge", ridge_with), ("Lasso", lasso_with)):
    bv_lam[name] = np.array([np.ravel(bias_variance(x, y, [deg_bv], fit=fit_with(lam))) for lam in lams_bv])

print(f"degree {deg_bv}, bias^2 with f:")
print(f"  OLS: error {ols_12[0]}, bias^2 {ols_12[2]}, var {ols_12[3]}")
for name in ("Ridge", "Lasso"):
    for lam, (err, b2, b2f, var) in list(zip(lams_bv, bv_lam[name]))[::2]:
        result = "not converged" if np.isnan(err) else f"error {err}, bias^2 {b2f}, var {var}"
        print(f"  {name} lambda {lam}: {result}")

In [ ]:
colors_m = {"OLS": "#777777", "Ridge": "#004488", "Lasso": "#BB5566"}

fig, ax = plt.subplots()
for name, (err, b2, b2f, var) in bv_methods.items():
    label = name if name == "OLS" else rf"{name}, $\lambda=10^{{{np.log10(lam_bv)}}}$"
    ax.semilogy(degrees_bv, b2f, "x-", color=colors_m[name], label=rf"bias$^2$, {label}")
    ax.semilogy(degrees_bv, var, "^--", color=colors_m[name], label=f"variance, {label}")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("bootstrap estimate")
ax.set_title(r"solid: bias$^2$ (with $f$), dashed: variance")
ax.legend(**legend_style)
plt.tight_layout()
plt.show()

for name in ("Ridge", "Lasso"):
    err, b2, b2f, var = bv_lam[name].T
    fig, ax = plt.subplots()
    ax.loglog(lams_bv, err, "o-", color="#004488", label="test error")
    ax.loglog(lams_bv, b2f, "x-", color="#BB5566", label=r"bias$^2$ (with $f$)")
    ax.loglog(lams_bv, var, "^-", color="#DDAA33", label="variance")
    ax.axhline(ols_12[3], color="#DDAA33", ls=":", label="OLS variance")
    ax.axhline(ols_12[2], color="#BB5566", ls=":", label=r"OLS bias$^2$")
    ax.set_xlabel(r"$\lambda$"); ax.set_ylabel("bootstrap estimate")
    ax.set_title(f"{name}, degree {deg_bv}")
    ax.legend(**legend_style)
    plt.tight_layout()
    plt.show()

**Extra checks for the report, part i):** uncertainty of the cross-validated selection, and the report figures.

In [ ]:
"""
How certain is the model selection: fold-to-fold spread at the best models, paired differences on the same
folds, the one-standard-error rule (Hastie et al., Sec. 7.10) and the selection with other fold assignments
"""
best_models = {}
# Tool: OpenAI chatgpt
# Role: Used to get inspiration to get a compact loop
for name, fit, mse in (("OLS", ols_fit, mse_ols[:, None]), ("Ridge", ridge_fit, mse_ridge), ("Lasso", lasso_fit, mse_lasso)):
    i, j = np.unravel_index(np.nanargmin(mse), mse.shape)
    lam  = 0.0 if name == "OLS" else lambda_parameters[j]

    fold_mse          = cv_folds(fit, x, y, polynomial_degrees[i], lam)
    best_models[name] = fold_mse
    print(f"{name:5s} (degree {polynomial_degrees[i]}, lambda {lam}): "
          f"{fold_mse.mean()} +- {fold_mse.std(ddof=1) / np.sqrt(len(fold_mse))} (SE)")

for a, b in (("Ridge", "OLS"), ("Lasso", "Ridge"), ("Lasso", "OLS")):
    difference = best_models[a] - best_models[b]
    print(f"{a} - {b}: {difference.mean():+.5f} +- {difference.std(ddof=1) / np.sqrt(len(difference))}")

folds_ols = np.array([cv_folds(ols_fit, x, y, degree) for degree in polynomial_degrees])
m_ols     = folds_ols.mean(axis=1)
se_ols    = folds_ols.std(axis=1, ddof=1) / np.sqrt(folds_ols.shape[1])
i         = np.argmin(m_ols)
print(f"OLS one-SE rule: degree {polynomial_degrees[np.argmax(m_ols <= m_ols[i] + se_ols[i])]} (minimum at {polynomial_degrees[i]})")

# other fold assignments; Lasso only for lambda >= 1e-6, where the solver mostly converges
lambdas_lasso = lambda_parameters[lambda_parameters >= 1e-6]
seed_runs     = {"OLS": [], "Ridge": [], "Lasso": []}
for seed in range(10):
    mse_o = np.array([cv_mse(ols_fit, x, y, degree, k=10, seed=seed) for degree in polynomial_degrees])
    mse_r = cv_grid(ridge_fit, 10, seed=seed)
    mse_l = cv_grid(lasso_fit, 10, lambdas=lambdas_lasso, seed=seed)

    i = np.argmin(mse_o)
    seed_runs["OLS"].append((polynomial_degrees[i], np.nan, mse_o[i]))
    for name, mse, lams in (("Ridge", mse_r, lambda_parameters), ("Lasso", mse_l, lambdas_lasso)):
        i, j = np.unravel_index(np.nanargmin(mse), mse.shape)
        seed_runs[name].append((polynomial_degrees[i], lams[j], mse[i, j]))

for name, runs in seed_runs.items():
    runs            = np.array(runs, dtype=float)
    seed_runs[name] = runs
    print(f"{name:5s}: degrees {runs[:, 0].astype(int)}, log10 lambda {np.round(np.log10(runs[:, 1]), 1)}, "
          f"MSE {runs[:, 2].mean()} +- {runs[:, 2].std(ddof=1)}")

for name, (err, b2, b2f, var) in bv_methods.items():
    values = ", ".join(f"deg {d}: {b2f[d - 1]}/{var[d - 1]}" for d in (4, 8, 12, 15))
    print(f"{name:5s} (bias^2/var): {values}")


"""
Report figures
"""
with plt.rc_context(report_style):
    fig, ax = plt.subplots(figsize=col)
    for k, ls in ((10, "-"), (5, "--")):
        m_o, m_r, m_l = results[k]
        label = (lambda s: s) if k == 10 else (lambda s: None)
        ax.semilogy(polynomial_degrees, m_o, "o" + ls, color="#777777", label=label("OLS"))
        ax.semilogy(polynomial_degrees, m_r.min(axis=1), "x" + ls, color="#004488", label=label(r"Ridge, best $\lambda$"))
        ax.semilogy(polynomial_degrees, np.nanmin(m_l, axis=1), "^" + ls, color="#BB5566", label=label(r"Lasso, best $\lambda$"))
    ax.axhline(sigma**2, color="k", ls=":", label=r"$\sigma^2$")
    ax.set_xlabel("polynomial degree")
    ax.set_ylabel("CV MSE")
    legend_outside(fig, ax, ncol=3, extra=((Line2D([], [], color="gray", ls="-"), "$k=10$"),
                                          (Line2D([], [], color="gray", ls="--"), "$k=5$")))
    save_report_fig(fig, "cv_degree")
    plt.show()

    # common colour scale, one colour bar
    fig, axes = plt.subplots(1, 2, figsize=wide2, sharey=True, constrained_layout=True)
    for ax, mse, name in ((axes[0], mse_ridge, "Ridge"), (axes[1], mse_lasso, "Lasso")):
        im   = ax.pcolormesh(np.log10(lambda_parameters), polynomial_degrees, np.log10(mse), shading="nearest", cmap="viridis",
                             vmin=np.log10(np.nanmin(mse_ridge)), vmax=-0.5)
        i, j = np.unravel_index(np.nanargmin(mse), mse.shape)
        ax.plot(np.log10(lambda_parameters[j]), polynomial_degrees[i], "r*", ms=9)
        ax.set_xlabel(r"$\log_{10}\lambda$")
        ax.set_title(f"{name}, 10-fold CV" + (" (white: not converged)" if name == "Lasso" else ""))
        ax.grid(False)
    axes[0].set_ylabel("polynomial degree")
    fig.colorbar(im, ax=axes.tolist(), label=r"$\log_{10}$ CV MSE", extend="max")
    save_report_fig(fig, "cv_heatmaps", tight=False)
    plt.show()

    fig, ax = plt.subplots(figsize=col)
    for name, (err, b2, b2f, var) in bv_methods.items():
        ax.semilogy(degrees_bv, b2f, "x-", color=colors_m[name], label=rf"bias$^2$, {name}")
        ax.semilogy(degrees_bv, var, "^--", color=colors_m[name], label=f"variance, {name}")
    ax.set_xlabel("polynomial degree")
    ax.set_ylabel("bootstrap estimate")
    ax.set_title(rf"$\lambda={lam_bv}$ for Ridge and Lasso")
    legend_outside(fig, ax, ncol=3)
    save_report_fig(fig, "bv_degree")
    plt.show()

    # Tool: OpenAI chatgpt
    # Role: Used to get cool colors for the plot
    fig, axes = plt.subplots(1, 2, figsize=wide2, sharey=True)
    for ax, name in ((axes[0], "Ridge"), (axes[1], "Lasso")):
        err, b2, b2f, var = bv_lam[name].T
        ax.loglog(lams_bv, err, "o-", color="#004488", label="test error")
        ax.loglog(lams_bv, b2f, "x-", color="#BB5566", label=r"bias$^2$ (with $f$)")
        ax.loglog(lams_bv, var, "^-", color="#DDAA33", label="variance")
        ax.axhline(ols_12[3], color="#DDAA33", ls=":", label="OLS variance")
        ax.axhline(ols_12[2], color="#BB5566", ls=":", label=r"OLS bias$^2$")
        ax.set_xlabel(r"$\lambda$")
        ax.set_title(f"{name}, degree {deg_bv}")
    axes[0].set_ylabel("bootstrap estimate")
    legend_outside(fig, axes, where="above", ncol=5)
    save_report_fig(fig, "bv_lambda")
    plt.show()

## Background literature

- The lecture notes of the course, in particular Chapter 2 (Sections 2.9–2.12
  on training and test error, the bias-variance tradeoff, the bootstrap and
  cross-validation), Chapter 3 (linear regression, Ridge, Lasso, scaling) and
  Chapter 4 (gradient descent methods and automatic differentiation).
- For a discussion and derivation of the variances and mean squared errors
  using linear regression, see the
  [Lecture notes on ridge regression by Wessel N. van Wieringen](https://arxiv.org/abs/1509.09169).
- The textbook of
  [Trevor Hastie, Robert Tibshirani, Jerome H. Friedman, The Elements of Statistical Learning, Springer](https://www.springer.com/gp/book/9780387848570);
  chapters 3 and 7 are the most relevant ones for the analysis here.
- On automatic differentiation:
  [Baydin et al., Automatic differentiation in machine learning: a survey, JMLR 18 (2018)](https://jmlr.org/papers/v18/17-468.html),
  and the [JAX documentation](https://jax.readthedocs.io).

## Introduction to numerical projects

Here follows a brief recipe and recommendation on how to answer the various
questions when preparing your answers.

- Give a short description of the nature of the problem and the eventual
  numerical methods you have used.
- Describe the algorithm you have used and/or developed. Here you may find it
  convenient to use pseudocoding. In many cases you can describe the algorithm
  in the program itself.
- Include the source code of your program. Comment your program properly. You
  should have the code at your GitHub/GitLab link. You can also place the code
  in an appendix of your report.
- If possible, try to find analytic solutions, or known limits in order to test
  your program when developing the code. In this project the closed-form OLS
  and Ridge solutions are exactly such tests for your gradient descent codes.
- Include your results either in figure form or in a table. Remember to label
  your results. All tables and figures should have relevant captions and labels
  on the axes.
- Try to evaluate the reliability and numerical stability/precision of your
  results. If possible, include a qualitative and/or quantitative discussion of
  the numerical stability, eventual loss of precision etc.
- Try to give an interpretation of your results in your answers to the
  problems.
- Critique: if possible include your comments and reflections about the
  exercise, whether you felt you learnt something, ideas for improvements and
  other thoughts you've made when solving the exercise. We wish to keep this
  course at the interactive level and your comments can help us improve it.
- Try to establish a practice where you log your work at the computer lab. You
  may find such a logbook very handy at later stages in your work, especially
  when you don't properly remember what a previous test version of your program
  did. Here you could also record the time spent on solving the exercise,
  various algorithms you may have tested or other topics which you feel worthy
  of mentioning.

## Format for electronic delivery of report and programs

The preferred format for the report is a PDF file. You can also use DOC or
postscript formats or an IPython notebook file. As programming language we
prefer that you choose between C/C++, Fortran2008, Julia or Python. The
following prescription should be followed when preparing the report:

- Use Canvas to hand in your projects, log in at
  <https://www.uio.no/english/services/it/education/canvas/> with your normal
  UiO username and password.
- Upload **only** the report file or the link to your GitHub/GitLab or similar
  type of repository! For the source code file(s) you have developed please
  provide us with your link to your GitHub/GitLab or similar domain. The report
  file should include all of your discussions and a list of the codes you have
  developed. Do not include library files which are available at the course
  homepage, unless you have made specific changes to them.
- In your GitHub/GitLab or similar repository, please include a folder which
  contains selected results. These can be in the form of output from your code
  for a selected set of runs and input parameters.

Finally, we encourage you to collaborate. Optimal working groups consist of 2–3
students. You can then hand in a common report.

## Software and needed installations

If you have Python installed (we recommend Python 3) and you feel pretty
familiar with installing different packages, we recommend that you install the
following Python packages via `pip` as

    pip install numpy scipy matplotlib ipython jupyter scikit-learn pandas jax jaxlib

(`autograd` is a lighter alternative to JAX for the automatic differentiation in
parts e)–g)). For Python 3, replace `pip` with `pip3` where needed.

For OSX users we recommend also, after having installed Xcode, to install
`brew`. Brew allows for a seamless installation of additional software via for
example `brew install python3`. For Linux users, with its variety of
distributions like for example the widely popular Ubuntu distribution, you can
use `pip` as well and simply install Python as `sudo apt-get install python3`
etc.

If you don't want to install various Python packages with their dependencies
separately, we recommend [Anaconda](https://docs.anaconda.com/), an open source
distribution of the Python and R programming languages for large-scale data
processing, predictive analytics, and scientific computing, that aims to
simplify package management and deployment. Package versions are managed by the
package management system `conda`.

Popular software packages written in Python for ML are
[Scikit-learn](http://scikit-learn.org/stable/), [JAX](https://jax.readthedocs.io),
[TensorFlow](https://www.tensorflow.org/), [PyTorch](http://pytorch.org/) and
[Keras](https://keras.io/). These are all freely available at their respective
GitHub sites. They encompass communities of developers in the thousands or more,
and the number of code developers and contributors keeps increasing.